---
## ABLATION & COMPARISON STUDIES

Test key components and benchmark against published models.


# StageBridge V1: GRANULAR Visual Pipeline

**Watch every step with figures!**

This notebook runs the full pipeline with visualizations at EVERY step:
- Synthetic data generation with ground truth visualization
- Stage distributions and donor structure
- Niche influence ground truth
- Training progress with loss curves (live)
- Latent space projections per epoch
- Ground truth recovery metrics

In [ ]:
# ============================================================================
# NATURE PUBLICATION-QUALITY VISUALIZATION SETUP
# ============================================================================
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
import json
import torch
import warnings
from IPython.display import display, clear_output
from datetime import datetime
from scipy.stats import gaussian_kde
from scipy.spatial import ConvexHull

# Path setup
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

# ============================================================================
# NATURE-QUALITY COLOR PALETTE (Deeper, Richer Colors)
# ============================================================================
# Stage progression colors - saturated and distinct
STAGE_COLORS = {
    "Normal": "#00A650",   # Deep green (healthy)
    "AAH": "#E63946",      # Deep red (early lesion)
    "AIS": "#457B9D",      # Deep blue (intermediate)
    "MIA": "#F77F00",      # Deep orange (late precursor)
    "LUAD": "#9C6644",     # Deep brown (invasive carcinoma)
    "Unknown": "#6C757D",  # Slate gray
}
STAGE_ORDER = ["Normal", "AAH", "AIS", "MIA", "LUAD"]

# Extended color palette for Nature figures
NATURE_PALETTE = {
    "primary_blue": "#1E3A8A",    # Deep blue
    "secondary_red": "#B91C1C",    # Deep red
    "tertiary_green": "#047857",   # Deep emerald
    "accent_orange": "#EA580C",    # Deep orange
    "accent_purple": "#7C3AED",    # Deep purple
    "neutral_dark": "#1F2937",     # Charcoal
    "neutral_mid": "#6B7280",      # Slate
    "neutral_light": "#D1D5DB",    # Light gray
    "background_white": "#FFFFFF",  # Pure white
    "grid": "#E5E7EB",             # Very light gray for grids
}

# Colorblind-friendly sequential palettes
SEQUENTIAL_BLUE = ['#EFF6FF', '#DBEAFE', '#BFDBFE', '#93C5FD', '#60A5FA', '#3B82F6', '#2563EB', '#1D4ED8', '#1E40AF', '#1E3A8A']
SEQUENTIAL_RED = ['#FEF2F2', '#FEE2E2', '#FECACA', '#FCA5A5', '#F87171', '#EF4444', '#DC2626', '#B91C1C', '#991B1B', '#7F1D1D']
SEQUENTIAL_GREEN = ['#ECFDF5', '#D1FAE5', '#A7F3D0', '#6EE7B7', '#34D399', '#10B981', '#059669', '#047857', '#065F46', '#064E3B']

# Diverging palette (for correlation matrices, heatmaps)
DIVERGING_PALETTE = sns.diverging_palette(240, 10, n=11, s=90, l=50, as_cmap=False)

# ============================================================================
# NATURE PUBLICATION MATPLOTLIB CONFIGURATION
# ============================================================================
mpl.rcParams.update({
    # Figure
    'figure.facecolor': '#FFFFFF',
    'figure.dpi': 150,                    # High-res display
    'figure.titlesize': 16,
    'figure.titleweight': 'bold',
    'figure.constrained_layout.use': True,

    # Axes
    'axes.facecolor': '#FFFFFF',
    'axes.edgecolor': '#1F2937',          # Dark charcoal edges
    'axes.labelcolor': '#1F2937',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.labelweight': 'normal',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 1.5,                # Thicker axes
    'axes.grid': False,
    'axes.axisbelow': True,               # Grid behind data

    # Font (DejaVu Sans is Nature-acceptable, similar to Helvetica)
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Liberation Sans'],
    'font.size': 11,
    'font.weight': 'normal',

    # Lines
    'lines.linewidth': 2.0,               # Thicker lines
    'lines.markersize': 6,
    'lines.markeredgewidth': 0.5,

    # Patches (scatter points, bars)
    'patch.linewidth': 0.5,
    'patch.edgecolor': '#1F2937',

    # Ticks
    'xtick.color': '#1F2937',
    'ytick.color': '#1F2937',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.minor.width': 1.0,
    'ytick.minor.width': 1.0,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.direction': 'out',
    'ytick.direction': 'out',

    # Grid
    'grid.color': '#E5E7EB',
    'grid.alpha': 0.5,
    'grid.linewidth': 0.8,
    'grid.linestyle': '--',

    # Legend
    'legend.frameon': True,
    'legend.framealpha': 1.0,
    'legend.facecolor': '#FFFFFF',
    'legend.edgecolor': '#D1D5DB',
    'legend.fontsize': 10,
    'legend.title_fontsize': 11,
    'legend.borderpad': 0.5,
    'legend.labelspacing': 0.5,
    'legend.handlelength': 2.0,
    'legend.handleheight': 0.7,
    'legend.handletextpad': 0.8,
    'legend.borderaxespad': 0.5,
    'legend.columnspacing': 2.0,

    # Saving (300 DPI for publication)
    'savefig.dpi': 300,
    'savefig.facecolor': '#FFFFFF',
    'savefig.edgecolor': 'none',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'savefig.format': 'png',

    # Images
    'image.cmap': 'viridis',
    'image.interpolation': 'bilinear',
})

# ============================================================================
# SEABORN CONFIGURATION (for advanced statistical plots)
# ============================================================================
sns.set_palette([STAGE_COLORS[s] for s in STAGE_ORDER])
sns.set_context("paper", font_scale=1.2)

# ============================================================================
# UTILITY FUNCTIONS FOR NATURE-QUALITY FIGURES
# ============================================================================

def save_figure(fig, name, formats=['png', 'pdf']):
    """Save figure in multiple formats for publication."""
    output_dir = Path("figures")
    output_dir.mkdir(exist_ok=True)
    for fmt in formats:
        path = output_dir / f"{name}.{fmt}"
        fig.savefig(path, dpi=300 if fmt=='png' else None,
                   facecolor='white', edgecolor='none', bbox_inches='tight')
    print(f"Saved: {name} ({', '.join(formats)})")

def add_panel_label(ax, label, x=-0.15, y=1.05, fontsize=18, fontweight='bold'):
    """Add Nature-style panel labels (A, B, C, etc.)."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=fontsize, fontweight=fontweight,
            va='top', ha='right')

def add_significance_bar(ax, x1, x2, y, h, text='***', fontsize=10):
    """Add significance bars for statistical comparisons."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c='black')
    ax.text((x1+x2)/2, y+h, text, ha='center', va='bottom', fontsize=fontsize)

def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=False):
    """Apply consistent Nature-style axis formatting."""
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=12, fontweight='normal')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=12, fontweight='normal')
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    if grid:
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

def add_density_contours(ax, x, y, colors='black', alpha=0.3, levels=5):
    """Add density contours to scatter plots (Nature style)."""
    from scipy.stats import gaussian_kde
    try:
        xy = np.vstack([x, y])
        z = gaussian_kde(xy)(xy)
        idx = z.argsort()
        x, y, z = x[idx], y[idx], z[idx]

        # Contour plot
        from scipy.interpolate import griddata
        xi = np.linspace(x.min(), x.max(), 100)
        yi = np.linspace(y.min(), y.max(), 100)
        xi, yi = np.meshgrid(xi, yi)
        zi = griddata((x, y), z, (xi, yi), method='cubic')

        ax.contour(xi, yi, zi, levels=levels, colors=colors,
                   alpha=alpha, linewidths=1.5)
    except Exception:
        pass  # Skip if contours fail

print("✓ Nature publication-quality visualization configured")
print(f"✓ Stage colors: {list(STAGE_COLORS.keys())}")
print(f"✓ DPI: {mpl.rcParams['savefig.dpi']} (publication quality)")
print(f"✓ Background: Pure white (#FFFFFF)")


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
from pathlib import Path
import torch

# Data paths
DATA_ROOT = Path.home() / "data" / "stagebridge" / "processed"
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "data" / "outputs" / "v1_demo"
OUTPUT_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

# Create directories
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Benchmark configuration
DIFFICULTY = "medium"  # small, medium, large
N_HVG = 2000          # Number of highly variable genes
LATENT_DIM = 128      # Latent space dimensionality
N_CELLS = 5000        # Total cells across all donors
N_DONORS = 5          # Number of synthetic donors/worlds
SEED = 42             # Random seed for reproducibility

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures directory: {FIGURES_DIR}")
print()
print(f"Benchmark difficulty: {DIFFICULTY}")
print(f"HVGs: {N_HVG}")
print(f"Latent dim: {LATENT_DIM}")
print(f"Total cells: {N_CELLS}")
print(f"Donors/worlds: {N_DONORS}")
print(f"Seed: {SEED}")
print(f"Device: {DEVICE}")
print("=" * 80)


---
## STEP 1: Generate Synthetic Data
Generate synthetic data with known ground truth for all 4 AGENTS.md suites

In [ ]:
# ============================================================================
# CELL 2: LOAD REFERENCE ATLASES FROM CENTRALIZED DATA
# ============================================================================
print("=" * 80)
print("LOADING REFERENCE ATLASES FROM CENTRALIZED DATA")
print("=" * 80)

from pathlib import Path
import scanpy as sc

# Centralized data directory
DATA_ROOT = Path.home() / "data" / "stagebridge" / "processed"
print(f"Data directory: {DATA_ROOT}\n")

# ============================================================================
# 1. HLCA (Human Lung Cell Atlas)
# ============================================================================
print("[1/3] HLCA (Human Lung Cell Atlas)")

HLCA_DIR = DATA_ROOT / "HLCA"
hlca_candidates = [
    HLCA_DIR / "hlca_scanvi.h5ad",      # From scvi-tools HubModel
    HLCA_DIR / "hlca_core.h5ad",        # From CellxGene direct
]

HLCA_PATH = None
hlca_adata = None
hlca_model = None

for path in hlca_candidates:
    if path.exists():
        HLCA_PATH = path
        size_mb = path.stat().st_size / (1024**2)
        print(f"  ✓ Found: {path.name} ({size_mb:.0f} MB)")
        
        # Load with backed mode for large files
        print(f"  Loading...")
        hlca_adata = sc.read_h5ad(HLCA_PATH, backed='r')
        print(f"  ✓ Loaded: {hlca_adata.n_obs:,} cells × {hlca_adata.n_vars:,} genes")
        
        # Check for scANVI latent
        if 'X_scANVI' in hlca_adata.obsm:
            print(f"  ✓ scANVI latent: {hlca_adata.obsm['X_scANVI'].shape[1]} dims")
        
        # Check annotations
        if 'cell_type' in hlca_adata.obs.columns:
            print(f"  ✓ Cell types: {hlca_adata.obs['cell_type'].nunique()}")
        elif 'ann_level_1' in hlca_adata.obs.columns:
            print(f"  ✓ Annotations: {hlca_adata.obs['ann_level_1'].nunique()}")
        
        break

if HLCA_PATH is None:
    print(f"  ✗ HLCA not found in {HLCA_DIR}")
    print(f"  To download:")
    print(f"    1. Run: python scripts/download_references.py --hlca")
    print(f"    2. Or download manually from CellxGene")
    raise FileNotFoundError("HLCA required for benchmark generation")

# ============================================================================
# 2. LuCA (Lung Cancer Atlas)
# ============================================================================
print("\n[2/3] LuCA (Lung Cancer Atlas)")

LUCA_DIR = DATA_ROOT / "LuCA"
luca_candidates = [
    LUCA_DIR / "f678fb47-e51b-4dc5-b23f-f9df43a67ee5.h5ad",  # Original CellxGene ID
    LUCA_DIR / "luca_extended.h5ad",                          # Renamed version
    LUCA_DIR / "luca_core.h5ad",
]

LUCA_PATH = None
luca_adata = None

for path in luca_candidates:
    if path.exists():
        LUCA_PATH = path
        size_gb = path.stat().st_size / (1024**3)
        print(f"  ✓ Found: {path.name} ({size_gb:.1f} GB)")
        
        # Load with backed mode (LuCA is large!)
        print(f"  Loading in backed mode...")
        luca_adata = sc.read_h5ad(LUCA_PATH, backed='r')
        print(f"  ✓ Loaded: {luca_adata.n_obs:,} cells × {luca_adata.n_vars:,} genes")
        
        # Check annotations
        if 'cell_type' in luca_adata.obs.columns:
            print(f"  ✓ Cell types: {luca_adata.obs['cell_type'].nunique()}")
        if 'disease' in luca_adata.obs.columns:
            print(f"  ✓ Disease states: {luca_adata.obs['disease'].nunique()}")
        
        break

if LUCA_PATH is None:
    print(f"  ⚠ LuCA not found in {LUCA_DIR}")
    print(f"  Options:")
    print(f"    1. Download (16 GB): https://luca.icbi.at/")
    print(f"       - Get 'Extended Atlas' h5ad file")
    print(f"       - Save to: {LUCA_DIR}/")
    print(f"    2. Use HLCA cancer proxy (automatic fallback)")
    print()
    print(f"  Using HLCA cancer proxy for now...")
    
    # Use HLCA cancer cells as proxy
    if 'disease' in hlca_adata.obs.columns:
        cancer_mask = hlca_adata.obs['disease'].str.contains(
            'cancer|tumor|malignant', case=False, na=False, regex=True
        )
        n_cancer = cancer_mask.sum()
        print(f"  ✓ Found {n_cancer:,} cancer-related cells in HLCA")
        print(f"  Using as LuCA proxy for benchmark")
    else:
        print(f"  ⚠ No disease annotations in HLCA")
        print(f"  Benchmark will use HLCA only")

# ============================================================================
# 3. Evolutionary Lung snRNA-seq (Optional)
# ============================================================================
print("\n[3/3] Evolutionary Lung snRNA-seq (Peng/Kadara)")

EVO_DIR = DATA_ROOT / "luad_evo"
evo_candidates = [
    EVO_DIR / "snrna_merged.h5ad",  # Actual filename
    EVO_DIR / "cells.h5ad",
]

EVO_PATH = None
evo_adata = None

for path in evo_candidates:
    if path.exists():
        EVO_PATH = path
        size_gb = path.stat().st_size / (1024**3)
        print(f"  ✓ Found: {path.name} ({size_gb:.1f} GB)")
        
        # Load with backed mode
        print(f"  Loading in backed mode...")
        evo_adata = sc.read_h5ad(EVO_PATH, backed='r')
        print(f"  ✓ Loaded: {evo_adata.n_obs:,} cells × {evo_adata.n_vars:,} genes")
        
        # Check stage annotations
        if 'stage' in evo_adata.obs.columns:
            stages = sorted(evo_adata.obs['stage'].unique())
            print(f"  ✓ Stages: {stages}")
        
        break

if EVO_PATH is None:
    print(f"  ⚠ Evolutionary data not found in {EVO_DIR}")
    print(f"  This is optional - benchmark will use HLCA + LuCA only")

# ============================================================================
# Summary
# ============================================================================
print("\n" + "=" * 80)
print("DATA LOADING COMPLETE")
print("=" * 80)
print(f"HLCA: {HLCA_PATH.name} - {hlca_adata.n_obs:,} cells")
if LUCA_PATH:
    print(f"LuCA: {LUCA_PATH.name} - {luca_adata.n_obs:,} cells")
else:
    print(f"LuCA: Using HLCA cancer proxy")
if EVO_PATH:
    print(f"Evolutionary: {EVO_PATH.name} - {evo_adata.n_obs:,} cells")
else:
    print(f"Evolutionary: Not available (optional)")
print("=" * 80)


In [ ]:
# ============================================================================
# GENERATE SEMI-SYNTHETIC BENCHMARK
# ============================================================================
print("\n" + "=" * 80)
print("GENERATING SEMI-SYNTHETIC BENCHMARK DATA")
print("=" * 80)
print("Using REAL expression profiles from HLCA/LuCA/Evolutionary data")
print("with SYNTHETIC spatial structure and ground truth transitions")
print("=" * 80)

from stagebridge.benchmarks.semi_synthetic import (
    SemiSyntheticBenchmarkGenerator,
    BenchmarkConfig,
)

# Configure semi-synthetic benchmark
benchmark_config = BenchmarkConfig(
    benchmark_name=f"granular_{DIFFICULTY}",
    
    # Data sources (from previous cell)
    hlca_path=HLCA_PATH,
    luca_path=LUCA_PATH,
    progression_path=EVO_PATH,
    
    # Feature harmonization
    n_hvg=N_HVG,
    latent_dim=LATENT_DIM,
    
    # Stages (lung cancer progression)
    stages=["Normal", "AAH", "AIS", "MIA", "LUAD"],
    
    # Spatial structure
    cells_per_world=N_CELLS // N_DONORS,
    world_width=1000.0,
    world_height=1000.0,
    
    # Splits
    n_worlds_train=5,
    n_worlds_val=2,
    n_worlds_test=3,
    
    # Reproducibility
    seed=SEED,
    
    # Output
    output_dir=DATA_DIR,
)

print("\nBenchmark Configuration:")
print(f"  Mode: Semi-synthetic (real expression + synthetic spatial)")
print(f"  HLCA: {benchmark_config.hlca_path}")
print(f"  LuCA: {benchmark_config.luca_path}")
if benchmark_config.progression_path:
    print(f"  Evolutionary: {benchmark_config.progression_path}")
print(f"  Cells per world: {benchmark_config.cells_per_world:,}")
print(f"  Worlds: train={benchmark_config.n_worlds_train}, val={benchmark_config.n_worlds_val}, test={benchmark_config.n_worlds_test}")
print(f"  Stages: {benchmark_config.stages}")
print(f"  HVGs: {benchmark_config.n_hvg}")

# Generate benchmark
print("\nGenerating benchmark (this may take 5-10 minutes)...")
print("-" * 80)

generator = SemiSyntheticBenchmarkGenerator(benchmark_config)
report = generator.generate(use_fallback_if_missing=True)

print("\n" + "=" * 80)
print("BENCHMARK GENERATION COMPLETE")
print("=" * 80)
print(f"Success: {report.success}")
print(f"\nGenerated worlds:")
for split, count in report.worlds_generated.items():
    print(f"  {split}: {count} worlds")

if report.output_paths:
    print(f"\nExported {len(report.output_paths)} files")
    print(f"Output directory: {benchmark_config.output_dir / benchmark_config.benchmark_name}")

if report.warnings:
    print("\nWarnings:")
    for warning in report.warnings:
        print(f"  ⚠ {warning}")

print("=" * 80)


In [ ]:
# ============================================================================
# CELL 4: VERIFY BENCHMARK EXPORT
# ============================================================================
print("\n" + "=" * 80)
print("BENCHMARK EXPORT VERIFICATION")
print("=" * 80)

benchmark_dir = DATA_DIR / f"granular_{DIFFICULTY}"

if benchmark_dir.exists():
    print(f"\nBenchmark directory: {benchmark_dir}")
    print("\nStructure:")
    
    # Check splits
    for split in ["train", "val", "test"]:
        split_dir = benchmark_dir / split
        if split_dir.exists():
            worlds = list(split_dir.glob("world_*"))
            print(f"\n{split.upper()}: {len(worlds)} worlds")
            
            # Check first world contents
            if worlds:
                world_dir = worlds[0]
                print(f"  Example world: {world_dir.name}")
                for f in world_dir.iterdir():
                    if f.is_file():
                        size_mb = f.stat().st_size / (1024**2)
                        print(f"    - {f.name}: {size_mb:.2f} MB")
    
    # Check manifest
    manifest_path = benchmark_dir / "benchmark_manifest.json"
    if manifest_path.exists():
        import json
        with open(manifest_path) as f:
            manifest = json.load(f)
        print(f"\nManifest:")
        print(f"  Benchmark: {manifest['benchmark_name']}")
        print(f"  HVGs: {manifest['n_hvg']}")
        print(f"  Harmonized genes: {len(manifest.get('harmonized_genes', []))} genes")
        print(f"  Splits: {manifest['splits']}")
    
    print("\n" + "=" * 80)
    print("BENCHMARK EXPORT COMPLETE")
    print("=" * 80)
    print("\nThis benchmark includes:")
    print("  ✓ Real expression profiles from HLCA/LuCA/Evolutionary data")
    print("  ✓ Synthetic spatial coordinates and neighborhoods")
    print("  ✓ Ground truth interaction rules and stage labels")
    print("  ✓ Self-contained h5ad files with harmonized genes")
    print("\nShare the entire benchmark directory with collaborators.")
    print("They do NOT need to download the 16+ GB atlas files.")
    print("=" * 80)
else:
    print(f"\n⚠ Benchmark directory not found: {benchmark_dir}")
    print("Run Cell 3 to generate the benchmark first.")



In [ ]:
# ============================================================================
# CELL 5: BASELINE COMPARISON
# ============================================================================
print("\n" + "=" * 80)
print("BASELINE EVALUATION")
print("=" * 80)

from stagebridge.baselines import run_baseline_comparison
from pathlib import Path

benchmark_dir = DATA_DIR / f"granular_{DIFFICULTY}"
results_dir = OUTPUT_DIR / "baseline_results"

if benchmark_dir.exists():
    print(f"\nRunning baseline comparison on: {benchmark_dir}")
    print(f"Output directory: {results_dir}\n")
    
    # Run evaluation
    results_df = run_baseline_comparison(
        benchmark_dir=benchmark_dir,
        output_dir=results_dir,
        device=DEVICE,
    )
    
    print("\n" + "=" * 80)
    print("BASELINE COMPARISON RESULTS")
    print("=" * 80)
    print(results_df.to_string(index=False))
    print("\n" + "=" * 80)
    
    print("\nBaselines tested:")
    print("  1. PoolingMLP - Bag-of-cells (no structure)")
    print("  2. DeepSets - Permutation invariance only")
    print("  3. SetTransformer - Flat attention (no spatial)")
    print("  4. GraphSAGE - Spatial graph structure")
    print("\nCore claim: StageBridge should outperform all baselines by conditioning")
    print("on receiver-centered local niche context + dual-reference anchoring.")
else:
    print(f"\n⚠ Benchmark not found: {benchmark_dir}")
    print("Run Cell 3 to generate the benchmark first.")

In [ ]:
# ============================================================================
# ABLATION STUDIES - Test Key Components
# ============================================================================
print("\n" + "=" * 80)
print("ABLATION STUDIES")
print("=" * 80)
print("Testing what happens when we remove key StageBridge components")
print("=" * 80)

# Ablation configurations
ablations = {
    "Full Model": {
        "use_niche": True,
        "use_dual_reference": True,
        "use_transformer": True,
        "use_spatial": True,
        "description": "Complete StageBridge (baseline)"
    },
    "No Niche Context": {
        "use_niche": False,
        "use_dual_reference": True,
        "use_transformer": True,
        "use_spatial": True,
        "description": "Remove receiver-centered niche conditioning"
    },
    "Single Reference (HLCA only)": {
        "use_niche": True,
        "use_dual_reference": False,  # HLCA only
        "use_transformer": True,
        "use_spatial": True,
        "description": "Remove dual-reference (LuCA), keep HLCA only"
    },
    "Pooled Niche": {
        "use_niche": True,
        "use_dual_reference": True,
        "use_transformer": False,  # Use mean pooling instead
        "use_spatial": True,
        "description": "Replace transformer with mean pooling"
    },
    "No Spatial": {
        "use_niche": True,
        "use_dual_reference": True,
        "use_transformer": True,
        "use_spatial": False,
        "description": "Remove spatial distance weighting"
    },
}

print("\nAblations to test:")
for name, config in ablations.items():
    print(f"  {name}: {config['description']}")

print("\n" + "-" * 80)
print("NOTE: Ablation training requires full model implementation.")
print("This cell shows the experimental design. Implementation:")
print("  - stagebridge/pipelines/run_ablations.py")
print("  - stagebridge/transition_model/baselines.py")
print("\nExpected result: Removing ANY component should degrade performance")
print("=" * 80)


In [ ]:
# ============================================================================
# COMPARISON WITH PUBLISHED MODELS
# ============================================================================
print("\n" + "=" * 80)
print("COMPARISON WITH PUBLISHED MODELS")
print("=" * 80)

# Published models for spatial transcriptomics progression inference
published_models = {
    "Squidpy": {
        "type": "Spatial statistics",
        "paper": "Palla et al. Nature Methods 2022",
        "approach": "Spatial graphs + neighborhood analysis",
        "limitation": "No progression modeling, descriptive only"
    },
    "Tangram": {
        "type": "Spatial mapping",
        "paper": "Biancalani et al. Nature Methods 2021",
        "approach": "Map scRNA-seq to spatial coordinates",
        "limitation": "No temporal dynamics or transitions"
    },
    "CellRank": {
        "type": "Trajectory inference",
        "paper": "Lange et al. Nature Methods 2022",
        "approach": "RNA velocity + Markov chains",
        "limitation": "No spatial niche conditioning"
    },
    "TrajectoryNet": {
        "type": "Flow-based trajectory",
        "paper": "Tong et al. ICML 2020",
        "approach": "Continuous normalizing flows",
        "limitation": "No niche context, no dual references"
    },
    "Waddington-OT": {
        "type": "Optimal transport",
        "paper": "Schiebinger et al. Cell 2019",
        "approach": "OT between time points",
        "limitation": "Cross-sectional only, no spatial structure"
    },
}

print("\nPublished models for comparison:")
for name, info in published_models.items():
    print(f"\n{name} ({info['type']})")
    print(f"  Paper: {info['paper']}")
    print(f"  Approach: {info['approach']}")
    print(f"  Limitation: {info['limitation']}")

print("\n" + "=" * 80)
print("StageBridge Novel Contributions:")
print("=" * 80)
print("1. Receiver-centered niche conditioning (vs flat aggregation)")
print("2. Dual-reference anchoring (HLCA + LuCA) (vs single reference)")
print("3. Cross-sectional progression inference (vs longitudinal requirement)")
print("4. Spatial + expression integration (vs spatial OR expression)")
print("5. Flow matching with niche context (vs context-free flows)")
print("=" * 80)

print("\nNOTE: Direct comparison requires re-implementing published models")
print("or using their official code with our benchmark data.")
print("See stagebridge/benchmarks/ for standardized evaluation protocol.")
print("=" * 80)


In [ ]:
# ============================================================================
# CELL 4: LOAD AND INSPECT SEMI-SYNTHETIC DATA
# ============================================================================
print("\n" + "=" * 80)
print("LOADING SEMI-SYNTHETIC DATA")
print("=" * 80)

# Load the generated benchmark data
if report and report.success:
    # Semi-synthetic data successfully generated
    print("Loading semi-synthetic benchmark data...")
    
    # The semi-synthetic generator outputs to DATA_DIR with specific structure
    # Load the main dataframes
    cells_path = DATA_DIR / "cells.parquet"
    transitions_path = DATA_DIR / "transitions.parquet"
    ground_truth_path = DATA_DIR / "ground_truth.json"
    
    if cells_path.exists():
        cells_df = pd.read_parquet(cells_path)
        print(f"✓ Loaded cells: {len(cells_df):,} rows")
    else:
        # Try alternative formats
        cells_path = DATA_DIR / "cells.h5ad"
        if cells_path.exists():
            import anndata
            cells_adata = anndata.read_h5ad(cells_path)
            cells_df = cells_adata.obs.copy()
            # Add expression/embedding data
            if 'X_pca' in cells_adata.obsm:
                cells_df['z_fused'] = list(cells_adata.obsm['X_pca'])
            print(f"✓ Loaded cells from h5ad: {len(cells_df):,} rows")
        else:
            raise FileNotFoundError(f"Could not find cells data in {DATA_DIR}")
    
    if transitions_path.exists():
        transitions_df = pd.read_parquet(transitions_path)
        print(f"✓ Loaded transitions: {len(transitions_df):,} rows")
    else:
        print("⚠ No transitions file found, will generate from cells")
        transitions_df = None
    
    if ground_truth_path.exists():
        with open(ground_truth_path) as f:
            ground_truth = json.load(f)
        print(f"✓ Loaded ground truth: {len(ground_truth)} keys")
    else:
        print("⚠ No ground truth file found")
        ground_truth = {}
    
else:
    # Fallback: load fully synthetic data
    print("Loading fallback synthetic data...")
    
    cells_df = pd.read_parquet(DATA_DIR / "cells.parquet")
    transitions_df = pd.read_parquet(DATA_DIR / "transitions.parquet") if (DATA_DIR / "transitions.parquet").exists() else None
    
    gt_path = DATA_DIR / "ground_truth.parquet"
    if gt_path.exists():
        gt_df = pd.read_parquet(gt_path)
        ground_truth = gt_df.iloc[0].to_dict() if len(gt_df) > 0 else {}
    else:
        ground_truth = {}
    
    print(f"✓ Loaded cells: {len(cells_df):,} rows")
    if transitions_df is not None:
        print(f"✓ Loaded transitions: {len(transitions_df):,} rows")

# Inspect loaded data
print("\n" + "=" * 80)
print("DATA INSPECTION")
print("=" * 80)

print(f"\nCells DataFrame: {cells_df.shape}")
print(f"Columns: {list(cells_df.columns)}")

# Check key columns
required_cols = ['stage', 'cell_type', 'donor_id']
for col in required_cols:
    if col in cells_df.columns:
        print(f"✓ {col}: {cells_df[col].nunique()} unique values")
    else:
        print(f"✗ {col}: MISSING")

# Check for embeddings
if 'z_fused' in cells_df.columns:
    print(f"✓ z_fused embeddings available (dim={len(cells_df['z_fused'].iloc[0])})")
elif 'X_pca' in cells_df.columns:
    print(f"✓ X_pca embeddings available")
else:
    print(f"⚠ No embeddings found, may need to compute")

# Check spatial coordinates
if 'x_spatial' in cells_df.columns and 'y_spatial' in cells_df.columns:
    print(f"✓ Spatial coordinates available")
else:
    print(f"⚠ No spatial coordinates")

# Stage distribution
if 'stage' in cells_df.columns:
    print(f"\nStage Distribution:")
    stage_counts = cells_df['stage'].value_counts().sort_index()
    for stage, count in stage_counts.items():
        print(f"  {stage}: {count:,} cells ({count/len(cells_df)*100:.1f}%)")

# Donor distribution  
if 'donor_id' in cells_df.columns:
    print(f"\nDonor Distribution: {cells_df['donor_id'].nunique()} donors")
    donor_counts = cells_df.groupby('donor_id')['stage'].value_counts().unstack(fill_value=0)
    print(f"  Cells per donor: {cells_df['donor_id'].value_counts().describe()[['mean', 'std', 'min', 'max']].to_dict()}")

# Ground truth
if ground_truth:
    print(f"\nGround Truth Keys: {list(ground_truth.keys())}")
    if 'stage_centroids' in ground_truth:
        print(f"  ✓ Stage centroids available")
    if 'influence_vectors' in ground_truth:
        print(f"  ✓ Niche influence vectors available ({len(ground_truth['influence_vectors'])} cell types)")
    if 'drift_strength' in ground_truth:
        print(f"  ✓ Flow dynamics parameters available")

# Stage edges for progression
stage_edges = [
    ("Normal", "AAH"),
    ("AAH", "AIS"),
    ("AIS", "MIA"),
    ("MIA", "LUAD"),
]

print("\n" + "=" * 80)
print("✓ DATA LOADED AND INSPECTED")
print("=" * 80)

---
## STEP 2: Visualize Data Distribution

In [ ]:
# ============================================================================
# CELL 4: FIGURE - STAGE DISTRIBUTION (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Stage Distribution - Ridge Plot & Progression Graph")
print("="*80)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

# ===== Panel 1: Ridge Plot (Joy Plot) - Cell count distributions by stage =====
ax_ridge = fig.add_subplot(gs[0, :2])

# Get stages in canonical order
stages = [s for s in STAGE_ORDER if s in cells_df['stage'].unique()]
n_stages = len(stages)

# Create ridge plot
ridge_height = 0.8
for i, stage in enumerate(stages):
    stage_cells = cells_df[cells_df['stage'] == stage]
    
    # Use z_fused first dimension for distribution
    z_vals = np.stack(stage_cells['z_fused'].values)[:, 0]
    
    # KDE
    try:
        kde = gaussian_kde(z_vals, bw_method=0.3)
        x_range = np.linspace(z_vals.min() - 0.5, z_vals.max() + 0.5, 200)
        y_density = kde(x_range)
        y_density = y_density / y_density.max() * ridge_height  # Normalize
        
        # Fill and line
        color = STAGE_COLORS.get(stage, '#999999')
        ax_ridge.fill_between(x_range, i, i + y_density, alpha=0.6, color=color, linewidth=0)
        ax_ridge.plot(x_range, i + y_density, color=color, linewidth=1.5)
    except:
        pass
    
    # Stage label with count
    count = len(stage_cells)
    ax_ridge.text(-4.5, i + 0.35, f"{stage}", fontsize=11, fontweight='bold', 
                  color=STAGE_COLORS.get(stage, '#333'), ha='right', va='center')
    ax_ridge.text(-4.5, i + 0.1, f"(n={count})", fontsize=9, color='#666', ha='right', va='center')

ax_ridge.set_yticks([])
ax_ridge.set_xlabel('Latent Expression (PC1)', fontsize=12, fontweight='medium')
ax_ridge.set_title('Cell Distribution by Stage (Ridge Plot)', fontsize=13, fontweight='bold', pad=10)
ax_ridge.set_xlim(-5, 5)
ax_ridge.spines['left'].set_visible(False)
add_grid(ax_ridge, alpha=0.2)

# ===== Panel 2: Stage Progression Graph (Network-style) =====
ax_prog = fig.add_subplot(gs[0, 2])

# Get stage edges
stage_edges = ground_truth.get('stage_edges', [])

# Position stages in a circle with progression layout
n = len(stages)
angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
positions = {stage: (np.cos(angles[i]) * 1.5, np.sin(angles[i]) * 1.5) for i, stage in enumerate(stages)}

# Draw edges first (behind nodes)
for src, tgt in stage_edges:
    if src in positions and tgt in positions:
        x1, y1 = positions[src]
        x2, y2 = positions[tgt]
        ax_prog.annotate('', xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle='-|>', color='#555555', lw=2.5,
                                       connectionstyle='arc3,rad=0.1'))

# Draw nodes
for stage in stages:
    x, y = positions[stage]
    count = len(cells_df[cells_df['stage'] == stage])
    size = 800 + count / 5  # Scale by cell count
    color = STAGE_COLORS.get(stage, '#999999')
    
    ax_prog.scatter([x], [y], s=size, c=[color], edgecolor='white', linewidth=3, zorder=5)
    ax_prog.scatter([x], [y], s=size * 1.1, facecolor='none', edgecolor=color, linewidth=2, zorder=4)
    
    # Label
    ax_prog.text(x, y - 0.05, stage, ha='center', va='center', fontsize=10, fontweight='bold', color='white', zorder=6)
    ax_prog.text(x, y - 0.3, f"n={count}", ha='center', va='top', fontsize=8, color='#444', zorder=6)

ax_prog.set_xlim(-2.2, 2.2)
ax_prog.set_ylim(-2.2, 2.2)
ax_prog.set_aspect('equal')
ax_prog.axis('off')
ax_prog.set_title('Stage Progression Network', fontsize=13, fontweight='bold', pad=10)

# ===== Panel 3: Donors per Stage (Horizontal bar with gradient) =====
ax_donors = fig.add_subplot(gs[1, 0])

donors_per_stage = cells_df.groupby('stage')['donor_id'].nunique().reindex(stages)
y_pos = np.arange(len(stages))
colors = [STAGE_COLORS.get(s, '#999999') for s in stages]

bars = ax_donors.barh(y_pos, donors_per_stage.values, color=colors, edgecolor='white', linewidth=1.5, height=0.7)

for i, (v, bar) in enumerate(zip(donors_per_stage.values, bars)):
    ax_donors.text(v + 0.2, i, str(v), va='center', fontsize=10, fontweight='medium')

ax_donors.set_yticks(y_pos)
ax_donors.set_yticklabels(stages)
ax_donors.set_xlabel('Number of Donors', fontsize=11, fontweight='medium')
ax_donors.set_title('Donors per Stage', fontsize=12, fontweight='bold')
ax_donors.invert_yaxis()
add_grid(ax_donors, alpha=0.2)

# ===== Panel 4: Cells per Stage (Proportional donut) =====
ax_donut = fig.add_subplot(gs[1, 1])

stage_counts = cells_df['stage'].value_counts().reindex(stages).fillna(0)
colors = [STAGE_COLORS.get(s, '#999999') for s in stages]

wedges, texts, autotexts = ax_donut.pie(
    stage_counts.values, 
    labels=stages,
    colors=colors,
    autopct='%1.1f%%',
    pctdistance=0.75,
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2),
    textprops=dict(fontsize=10)
)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('medium')

# Center text
ax_donut.text(0, 0, f'{len(cells_df):,}\ncells', ha='center', va='center', fontsize=12, fontweight='bold')
ax_donut.set_title('Stage Composition', fontsize=12, fontweight='bold')

# ===== Panel 5: Summary Statistics =====
ax_stats = fig.add_subplot(gs[1, 2])
ax_stats.axis('off')

# Calculate statistics
stats_text = f"""
Stage Distribution Summary
{'─' * 30}

Total Cells: {len(cells_df):,}
Total Donors: {cells_df['donor_id'].nunique()}
Stages: {len(stages)}

Cells per Stage:
"""
for stage in stages:
    count = len(cells_df[cells_df['stage'] == stage])
    pct = count / len(cells_df) * 100
    stats_text += f"  {stage}: {count:,} ({pct:.1f}%)\n"

stats_text += f"""
Transitions: {len(transitions_df):,}
Difficulty: {DIFFICULTY.upper()}
"""

ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes, fontsize=10,
              verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))

plt.suptitle('Stage Distribution Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig1_stage_distribution")
plt.show()

In [ ]:
# ============================================================================
# CELL 5: FIGURE - DONOR STRUCTURE (Publication Quality - Clustered Heatmap)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Donor and Clone Structure - Clustered Heatmap")
print("="*80)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.35)

# ===== Panel 1: Clustered Heatmap (Stage x Donor) =====
ax_cluster = fig.add_subplot(gs[0, :2])

# Create stage x donor matrix
stage_donor = cells_df.groupby(['stage', 'donor_id']).size().unstack(fill_value=0)

# Reorder stages to canonical order
ordered_stages = [s for s in STAGE_ORDER if s in stage_donor.index]
stage_donor = stage_donor.reindex(ordered_stages)

# Row-normalize for better visualization
stage_donor_norm = stage_donor.div(stage_donor.sum(axis=1), axis=0)

# Draw heatmap with hierarchical clustering on columns (donors)
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist

# Cluster donors
if stage_donor.shape[1] > 2:
    donor_linkage = hierarchy.linkage(pdist(stage_donor.T), method='ward')
    donor_order = hierarchy.leaves_list(donor_linkage)
    stage_donor_ordered = stage_donor.iloc[:, donor_order]
else:
    stage_donor_ordered = stage_donor

im = ax_cluster.imshow(stage_donor_ordered, cmap='YlOrRd', aspect='auto', interpolation='nearest')

# Add cell counts as annotations
for i in range(len(ordered_stages)):
    for j in range(stage_donor_ordered.shape[1]):
        val = stage_donor_ordered.iloc[i, j]
        text_color = 'white' if val > stage_donor_ordered.values.max() * 0.6 else 'black'
        ax_cluster.text(j, i, str(int(val)), ha='center', va='center', fontsize=8, color=text_color)

ax_cluster.set_xticks(np.arange(stage_donor_ordered.shape[1]))
ax_cluster.set_xticklabels(stage_donor_ordered.columns, rotation=45, ha='right', fontsize=9)
ax_cluster.set_yticks(np.arange(len(ordered_stages)))
ax_cluster.set_yticklabels(ordered_stages, fontsize=10)

# Color the y-axis labels by stage
for i, stage in enumerate(ordered_stages):
    ax_cluster.get_yticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333333'))
    ax_cluster.get_yticklabels()[i].set_fontweight('bold')

cbar = fig.colorbar(im, ax=ax_cluster, shrink=0.8, pad=0.02)
cbar.set_label('Cell Count', fontsize=11, fontweight='medium')

ax_cluster.set_xlabel('Donor (clustered)', fontsize=11, fontweight='medium')
ax_cluster.set_ylabel('Stage', fontsize=11, fontweight='medium')
ax_cluster.set_title('Cell Distribution: Stage × Donor (Hierarchically Clustered)', fontsize=12, fontweight='bold')

# ===== Panel 2: Donor Composition Stacked Bar =====
ax_stack = fig.add_subplot(gs[0, 2])

# Stacked bar chart showing donor composition
donor_stage = cells_df.groupby(['donor_id', 'stage']).size().unstack(fill_value=0)
donor_stage = donor_stage.reindex(columns=ordered_stages)
donor_stage_pct = donor_stage.div(donor_stage.sum(axis=1), axis=0) * 100

# Sort donors by progression (more advanced stages later)
donor_order = donor_stage_pct.apply(
    lambda row: sum(i * row.iloc[i] for i in range(len(row))), axis=1
).sort_values().index

bottom = np.zeros(len(donor_order))
for stage in ordered_stages:
    if stage in donor_stage_pct.columns:
        vals = donor_stage_pct.loc[donor_order, stage].values
        color = STAGE_COLORS.get(stage, '#999999')
        ax_stack.barh(range(len(donor_order)), vals, left=bottom, color=color, 
                     label=stage, edgecolor='white', linewidth=0.5, height=0.8)
        bottom += vals

ax_stack.set_yticks(range(len(donor_order)))
ax_stack.set_yticklabels(donor_order, fontsize=9)
ax_stack.set_xlabel('Percentage (%)', fontsize=11, fontweight='medium')
ax_stack.set_ylabel('Donor', fontsize=11, fontweight='medium')
ax_stack.set_title('Stage Composition per Donor', fontsize=12, fontweight='bold')
ax_stack.set_xlim(0, 100)
style_legend(ax_stack, loc='upper right', title='Stage')

# ===== Panel 3: Cells per Donor (Lollipop) =====
ax_lollipop = fig.add_subplot(gs[1, 0])

donor_counts = cells_df['donor_id'].value_counts().sort_values(ascending=True)
y_pos = np.arange(len(donor_counts))

# Draw lollipop
ax_lollipop.hlines(y=y_pos, xmin=0, xmax=donor_counts.values, color='#0E7490', alpha=0.7, linewidth=2)
ax_lollipop.scatter(donor_counts.values, y_pos, color='#0E7490', s=80, zorder=3, edgecolor='white', linewidth=2)

for i, v in enumerate(donor_counts.values):
    ax_lollipop.text(v + 5, i, str(v), va='center', fontsize=9)

ax_lollipop.set_yticks(y_pos)
ax_lollipop.set_yticklabels(donor_counts.index, fontsize=9)
ax_lollipop.set_xlabel('Cell Count', fontsize=11, fontweight='medium')
ax_lollipop.set_title('Cells per Donor', fontsize=12, fontweight='bold')
add_grid(ax_lollipop, alpha=0.2)

# ===== Panel 4: Clones per Donor (Violin) =====
ax_violin = fig.add_subplot(gs[1, 1])

clones_per_donor = cells_df.groupby('donor_id')['clone_id'].nunique()

# Create violin plot data by stage for donors
donor_stage_primary = cells_df.groupby('donor_id')['stage'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown')

violin_data = []
violin_labels = []
for stage in ordered_stages:
    donors_in_stage = donor_stage_primary[donor_stage_primary == stage].index
    clones = clones_per_donor.loc[clones_per_donor.index.isin(donors_in_stage)]
    if len(clones) > 0:
        violin_data.append(clones.values)
        violin_labels.append(stage)

if violin_data:
    parts = ax_violin.violinplot(violin_data, positions=range(len(violin_labels)), showmeans=True, showmedians=True)
    
    # Color violins by stage
    for i, (pc, stage) in enumerate(zip(parts['bodies'], violin_labels)):
        pc.set_facecolor(STAGE_COLORS.get(stage, '#999999'))
        pc.set_alpha(0.7)
    
    # Style other elements
    for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
        if partname in parts:
            parts[partname].set_color('#333333')
            parts[partname].set_linewidth(1.5)

ax_violin.set_xticks(range(len(violin_labels)))
ax_violin.set_xticklabels(violin_labels, fontsize=10)
ax_violin.set_ylabel('Clones per Donor', fontsize=11, fontweight='medium')
ax_violin.set_xlabel('Primary Stage', fontsize=11, fontweight='medium')
ax_violin.set_title('Clone Diversity by Stage', fontsize=12, fontweight='bold')
add_grid(ax_violin, alpha=0.2)

# ===== Panel 5: Clone Size Distribution =====
ax_clone = fig.add_subplot(gs[1, 2])

clone_sizes = cells_df.groupby('clone_id').size()
ax_clone.hist(clone_sizes, bins=30, color='#7C3AED', edgecolor='white', linewidth=1, alpha=0.8)
ax_clone.axvline(clone_sizes.median(), color='#E63946', linestyle='--', linewidth=2, 
                label=f'Median: {clone_sizes.median():.0f}')
ax_clone.axvline(clone_sizes.mean(), color='#2A9D8F', linestyle=':', linewidth=2,
                label=f'Mean: {clone_sizes.mean():.1f}')

ax_clone.set_xlabel('Cells per Clone', fontsize=11, fontweight='medium')
ax_clone.set_ylabel('Frequency', fontsize=11, fontweight='medium')
ax_clone.set_title('Clone Size Distribution', fontsize=12, fontweight='bold')
style_legend(ax_clone, loc='upper right')
add_grid(ax_clone, alpha=0.2)

plt.suptitle('Donor and Clone Structure Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig2_donor_structure")
plt.show()

In [ ]:
# ============================================================================
# CELL 6: FIGURE - CELL TYPE DISTRIBUTION (Publication Quality - Ridge Plot)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Cell Type Distribution")
print("="*80)

fig = plt.figure(figsize=(18, 6))
gs = fig.add_gridspec(1, 3, wspace=0.3)

# ===== Panel 1: Cell count by type (sorted bar) =====
ax = fig.add_subplot(gs[0, 0])

ct_counts = cells_df['cell_type'].value_counts().sort_values(ascending=True)
y_pos = np.arange(len(ct_counts))

colors_ct = plt.cm.Set3(np.linspace(0, 1, len(ct_counts)))
bars = ax.barh(y_pos, ct_counts.values, color=colors_ct, edgecolor='white', linewidth=1)

# Add count labels
for i, v in enumerate(ct_counts.values):
    ax.text(v + max(ct_counts) * 0.02, i, str(v), va='center', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(ct_counts.index, fontsize=9)
ax.set_xlabel('Cell Count', fontsize=11, fontweight='medium')
ax.set_title('Cells per Cell Type', fontsize=12, fontweight='bold')
add_grid(ax, alpha=0.2)

# ===== Panel 2: Cell type by stage (stacked bar) =====
ax = fig.add_subplot(gs[0, 1])

stage_ct = cells_df.groupby(['stage', 'cell_type']).size().unstack(fill_value=0)
stage_ct = stage_ct.reindex([s for s in STAGE_ORDER if s in stage_ct.index])

# Normalize to percentages
stage_ct_pct = stage_ct.div(stage_ct.sum(axis=1), axis=0) * 100

# Plot stacked bars
bottom = np.zeros(len(stage_ct_pct))
ct_list = stage_ct_pct.columns[:8]  # Top 8 cell types
colors_ct = plt.cm.Set3(np.linspace(0, 1, len(ct_list)))

for i, ct in enumerate(ct_list):
    if ct in stage_ct_pct.columns:
        vals = stage_ct_pct[ct].values
        ax.bar(range(len(stage_ct_pct)), vals, bottom=bottom, 
              label=ct, color=colors_ct[i], edgecolor='white', linewidth=0.5)
        bottom += vals

ax.set_xticks(range(len(stage_ct_pct)))
ax.set_xticklabels(stage_ct_pct.index, fontsize=10)
ax.set_ylabel('Percentage (%)', fontsize=11, fontweight='medium')
ax.set_xlabel('Stage', fontsize=11, fontweight='medium')
ax.set_title('Cell Type Composition by Stage', fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
style_legend(ax, loc='upper right', title='Cell Type')

# ===== Panel 3: Diversity index by stage =====
ax = fig.add_subplot(gs[0, 2])

from scipy.stats import entropy

def simpson_diversity(counts):
    """Simpson's diversity index: 1 - sum(p_i^2)"""
    p = counts / counts.sum()
    return 1 - np.sum(p**2)

diversity_by_stage = []
stages_list = [s for s in STAGE_ORDER if s in cells_df['stage'].unique()]

for stage in stages_list:
    stage_cells = cells_df[cells_df['stage'] == stage]
    ct_counts = stage_cells['cell_type'].value_counts()
    diversity = simpson_diversity(ct_counts.values)
    diversity_by_stage.append(diversity)

bars = ax.bar(range(len(stages_list)), diversity_by_stage, 
             color=[STAGE_COLORS.get(s, '#999') for s in stages_list],
             edgecolor='white', linewidth=1.5, width=0.7)

# Add value labels
for i, (stage, div) in enumerate(zip(stages_list, diversity_by_stage)):
    ax.text(i, div + 0.02, f'{div:.3f}', ha='center', fontsize=10, fontweight='medium')

ax.set_xticks(range(len(stages_list)))
ax.set_xticklabels(stages_list, fontsize=10)
ax.set_ylabel("Simpson's Diversity Index", fontsize=11, fontweight='medium')
ax.set_xlabel('Stage', fontsize=11, fontweight='medium')
ax.set_title('Cell Type Diversity by Stage', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1)
add_grid(ax, alpha=0.2)

plt.suptitle('Cell Type Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig3_cell_type_distribution")
plt.show()

print(f"\nTotal cell types: {cells_df['cell_type'].nunique()}")
print(f"Mean diversity: {np.mean(diversity_by_stage):.3f}")

---
## STEP 3: Visualize Ground Truth

In [ ]:
# ============================================================================
# CELL 7: FIGURE - GROUND TRUTH: STAGE CENTROIDS (Suite A) - Publication Quality
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Stage Centroids with Advanced UMAP")
print("="*80)

# Get stage centroids from ground truth
stage_centroids = ground_truth.get('stage_centroids', {})

if stage_centroids:
    fig = plt.figure(figsize=(18, 6))
    gs = fig.add_gridspec(1, 3, wspace=0.3)
    
    # ===== Panel 1: Advanced scatter with density contours, convex hulls, confidence ellipses =====
    ax = fig.add_subplot(gs[0, 0])
    
    # Sample cells for visualization
    sample_cells = cells_df.sample(min(1000, len(cells_df)), random_state=42)
    stages_list = [s for s in STAGE_ORDER if s in sample_cells['stage'].unique()]
    
    # Draw density contours for overall distribution
    all_z = np.stack(sample_cells['z_fused'].values)
    draw_density_contours(ax, all_z[:, 0], all_z[:, 1], levels=6, alpha=0.15)
    
    # Plot each stage with hulls and ellipses
    for stage in stages_list:
        mask = sample_cells['stage'] == stage
        z_fused = np.stack(sample_cells.loc[mask, 'z_fused'].values)
        color = STAGE_COLORS.get(stage, '#999999')
        
        # Draw convex hull
        draw_convex_hull(ax, z_fused[:, :2], color, alpha=0.12)
        
        # Draw confidence ellipse (2σ)
        draw_confidence_ellipse(ax, z_fused[:, 0], z_fused[:, 1], n_std=2.0, 
                               color=color, alpha=0.5, linewidth=2)
        
        # Scatter points with white edge
        ax.scatter(z_fused[:, 0], z_fused[:, 1], s=25, c=[color], alpha=0.6, 
                  label=stage, edgecolor='white', linewidth=0.5, rasterized=True)
    
    # Draw centroids as stars
    centroids_2d = {stage: np.array(vec)[:2] for stage, vec in stage_centroids.items()}
    for stage, pos in centroids_2d.items():
        if stage in stages_list:
            color = STAGE_COLORS.get(stage, '#999999')
            ax.scatter(pos[0], pos[1], s=400, c=[color], marker='*', 
                      edgecolor='black', linewidth=1.5, zorder=10)
            ax.annotate(stage, (pos[0], pos[1]), fontsize=9, fontweight='bold',
                       xytext=(8, 8), textcoords='offset points',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Latent Dim 0', fontsize=11, fontweight='medium')
    ax.set_ylabel('Latent Dim 1', fontsize=11, fontweight='medium')
    ax.set_title('Stage Clusters: Hulls & Ellipses', fontsize=12, fontweight='bold')
    style_legend(ax, loc='best', title='Stage')
    add_grid(ax, alpha=0.15)
    
    # ===== Panel 2: Centroid Distance Matrix Heatmap =====
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Compute distance matrix between centroids
    n_stages = len(stages_list)
    dist_matrix = np.zeros((n_stages, n_stages))
    for i, s1 in enumerate(stages_list):
        for j, s2 in enumerate(stages_list):
            if s1 in centroids_2d and s2 in centroids_2d:
                dist_matrix[i, j] = np.linalg.norm(
                    np.array(stage_centroids.get(s1, [0]*LATENT_DIM)) - 
                    np.array(stage_centroids.get(s2, [0]*LATENT_DIM))
                )
    
    im = ax2.imshow(dist_matrix, cmap='YlOrRd', aspect='auto')
    
    # Annotate
    for i in range(n_stages):
        for j in range(n_stages):
            text_color = 'white' if dist_matrix[i, j] > dist_matrix.max() * 0.6 else 'black'
            ax2.text(j, i, f'{dist_matrix[i, j]:.2f}', ha='center', va='center', 
                    fontsize=9, color=text_color, fontweight='medium')
    
    ax2.set_xticks(range(n_stages))
    ax2.set_xticklabels(stages_list, rotation=45, ha='right')
    ax2.set_yticks(range(n_stages))
    ax2.set_yticklabels(stages_list)
    
    # Color labels
    for i, stage in enumerate(stages_list):
        ax2.get_xticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))
        ax2.get_yticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))
    
    cbar = fig.colorbar(im, ax=ax2, shrink=0.8)
    cbar.set_label('Euclidean Distance', fontsize=10)
    ax2.set_title('Centroid Distance Matrix', fontsize=12, fontweight='bold')
    
    # ===== Panel 3: Stage Separation (Silhouette-style) =====
    ax3 = fig.add_subplot(gs[0, 2])
    
    # Compute inter-stage vs intra-stage distances
    inter_dists = []
    intra_dists = []
    stage_names = []
    
    for stage in stages_list:
        mask = sample_cells['stage'] == stage
        z_fused = np.stack(sample_cells.loc[mask, 'z_fused'].values)
        
        if len(z_fused) > 1:
            # Intra-cluster distances
            centroid = np.mean(z_fused, axis=0)
            intra = np.mean(np.linalg.norm(z_fused - centroid, axis=1))
            intra_dists.append(intra)
            
            # Inter-cluster distance (to nearest different cluster)
            other_centroids = [np.array(stage_centroids[s]) for s in stages_list if s != stage and s in stage_centroids]
            if other_centroids:
                inter = np.min([np.linalg.norm(centroid - oc) for oc in other_centroids])
            else:
                inter = 0
            inter_dists.append(inter)
            stage_names.append(stage)
    
    x = np.arange(len(stage_names))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, intra_dists, width, label='Intra-cluster', 
                   color=[STAGE_COLORS.get(s, '#999') for s in stage_names], alpha=0.7, edgecolor='white')
    bars2 = ax3.bar(x + width/2, inter_dists, width, label='Inter-cluster (min)',
                   color='#333333', alpha=0.8, edgecolor='white')
    
    # Add separation ratio as text
    for i, (intra, inter) in enumerate(zip(intra_dists, inter_dists)):
        ratio = inter / (intra + 1e-8)
        ax3.text(i, max(intra, inter) + 0.1, f'×{ratio:.1f}', ha='center', fontsize=9, fontweight='bold')
    
    ax3.set_xticks(x)
    ax3.set_xticklabels(stage_names)
    ax3.set_ylabel('Distance', fontsize=11, fontweight='medium')
    ax3.set_title('Stage Separation Quality', fontsize=12, fontweight='bold')
    style_legend(ax3, loc='upper right')
    add_grid(ax3, alpha=0.2)
    
    plt.suptitle('Ground Truth: Stage Centroids (Suite A - Flow Field)', fontsize=14, fontweight='bold', y=1.02)
    save_figure(fig, OUTPUT_DIR, "fig4_stage_centroids")
    plt.show()
else:
    print("No stage centroids in ground truth")

In [ ]:
# ============================================================================
# CELL 8: FIGURE - GROUND TRUTH: NICHE INFLUENCE (Suite B) - Radar Chart
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Niche Influence Vectors (Radar Chart)")
print("="*80)

influence_vectors = ground_truth.get('influence_vectors', {})
influential_cts = ground_truth.get('influential_celltypes', [])

if influence_vectors:
    fig = plt.figure(figsize=(18, 6))
    gs = fig.add_gridspec(1, 3, wspace=0.35)
    
    # ===== Panel 1: Radar/Spider Chart - Influence Vector Dimensions =====
    ax_radar = fig.add_subplot(gs[0, 0], projection='polar')
    
    # Use first 8 dimensions for radar
    n_dims = min(8, LATENT_DIM)
    angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
    angles += angles[:1]  # Close the polygon
    
    cts = list(influence_vectors.keys())[:6]  # Top 6 cell types
    colors = plt.cm.Set2(np.linspace(0, 1, len(cts)))
    
    for idx, ct in enumerate(cts):
        # FIX: Convert list to numpy array
        vec = np.array(influence_vectors[ct][:n_dims])
        # Normalize to [0, 1]
        vec_norm = (vec - vec.min()) / (vec.max() - vec.min() + 1e-8)
        vec_plot = vec_norm.tolist() + vec_norm[:1].tolist()
        
        ax_radar.plot(angles, vec_plot, 'o-', linewidth=2, label=ct, color=colors[idx], alpha=0.8)
        ax_radar.fill(angles, vec_plot, alpha=0.15, color=colors[idx])
    
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels([f'd{i}' for i in range(n_dims)], size=10)
    ax_radar.set_title('Influence Vector Profile (Radar)', fontsize=12, fontweight='bold', pad=20)
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.0), fontsize=9)
    ax_radar.grid(True, linestyle='--', alpha=0.3)
    
    # ===== Panel 2: Influence Magnitude Bar + Violin =====
    ax_bar = fig.add_subplot(gs[0, 1])
    
    # Compute magnitudes - FIX: Convert to numpy arrays
    magnitudes = {ct: np.linalg.norm(np.array(vec)) for ct, vec in influence_vectors.items()}
    sorted_cts = sorted(magnitudes.keys(), key=lambda x: magnitudes[x], reverse=True)
    
    y_pos = np.arange(len(sorted_cts))
    mags = [magnitudes[ct] for ct in sorted_cts]
    
    # Color influential cell types differently
    bar_colors = ['#E63946' if ct in influential_cts else '#457B9D' for ct in sorted_cts]
    
    bars = ax_bar.barh(y_pos, mags, color=bar_colors, edgecolor='white', linewidth=1.5, height=0.7)
    
    # Add magnitude values
    for i, (v, ct) in enumerate(zip(mags, sorted_cts)):
        marker = '★' if ct in influential_cts else ''
        ax_bar.text(v + 0.05, i, f'{v:.2f} {marker}', va='center', fontsize=9)
    
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(sorted_cts, fontsize=10)
    ax_bar.set_xlabel('Influence Magnitude ||v||', fontsize=11, fontweight='medium')
    ax_bar.set_title('Cell Type Influence Magnitudes', fontsize=12, fontweight='bold')
    ax_bar.invert_yaxis()
    
    # Add legend for colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#E63946', label='Influential (★)'),
                       Patch(facecolor='#457B9D', label='Other')]
    ax_bar.legend(handles=legend_elements, loc='lower right', fontsize=9)
    add_grid(ax_bar, alpha=0.2)
    
    # ===== Panel 3: Niche Influence Score Distribution by Stage =====
    ax_dist = fig.add_subplot(gs[0, 2])
    
    stages_list = [s for s in STAGE_ORDER if s in cells_df['stage'].unique()]
    
    # Violin plot of niche influence score by stage
    violin_data = [cells_df[cells_df['stage'] == s]['niche_influence_score'].values for s in stages_list]
    
    parts = ax_dist.violinplot(violin_data, positions=range(len(stages_list)), 
                               showmeans=True, showmedians=True, widths=0.8)
    
    # Color violins by stage
    for i, (pc, stage) in enumerate(zip(parts['bodies'], stages_list)):
        pc.set_facecolor(STAGE_COLORS.get(stage, '#999999'))
        pc.set_alpha(0.7)
        pc.set_edgecolor('white')
    
    # Style other elements
    for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
        if partname in parts:
            parts[partname].set_color('#333333')
            parts[partname].set_linewidth(1.5)
    
    # Overlay individual points (jittered)
    for i, (data, stage) in enumerate(zip(violin_data, stages_list)):
        jitter = np.random.normal(0, 0.08, len(data))
        ax_dist.scatter(i + jitter, data, s=10, alpha=0.3, 
                       c=[STAGE_COLORS.get(stage, '#999')], edgecolor='none')
    
    ax_dist.set_xticks(range(len(stages_list)))
    ax_dist.set_xticklabels(stages_list, fontsize=10)
    ax_dist.set_ylabel('Niche Influence Score', fontsize=11, fontweight='medium')
    ax_dist.set_xlabel('Stage', fontsize=11, fontweight='medium')
    ax_dist.set_title('Niche Influence by Stage', fontsize=12, fontweight='bold')
    add_grid(ax_dist, alpha=0.2)
    
    # Add mean annotations
    for i, data in enumerate(violin_data):
        mean_val = np.mean(data)
        ax_dist.annotate(f'{mean_val:.2f}', (i, mean_val), xytext=(5, 5), 
                        textcoords='offset points', fontsize=8, fontweight='medium')
    
    plt.suptitle('Ground Truth: Niche Influence (Suite B)', fontsize=14, fontweight='bold', y=1.02)
    save_figure(fig, OUTPUT_DIR, "fig5_niche_influence_gt")
    plt.show()
    
    print(f"\nInfluential cell types: {influential_cts}")
    print(f"Top influence magnitudes: {dict(list(sorted(magnitudes.items(), key=lambda x: -x[1]))[:5])}")
else:
    print("No influence vectors in ground truth")

In [ ]:
# ============================================================================
# CELL 9: FIGURE - GROUND TRUTH: TRANSITIONS (Suite A) - Sankey Diagram
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Ground Truth - Transition Dynamics & Sankey")
print("="*80)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

# ===== Panel 1: Transition Flow Matrix (Heatmap) =====
ax_flow = fig.add_subplot(gs[0, 0])

# Build transition matrix
stages_list = [s for s in STAGE_ORDER if s in cells_df['stage'].unique()]
trans_matrix = np.zeros((len(stages_list), len(stages_list)))

for i, src in enumerate(stages_list):
    for j, tgt in enumerate(stages_list):
        mask = (transitions_df['source_stage'] == src) & (transitions_df['target_stage'] == tgt)
        trans_matrix[i, j] = mask.sum()

# Normalize by row
trans_matrix_norm = trans_matrix / (trans_matrix.sum(axis=1, keepdims=True) + 1e-8)

im = ax_flow.imshow(trans_matrix_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)

# Annotations
for i in range(len(stages_list)):
    for j in range(len(stages_list)):
        val = trans_matrix_norm[i, j]
        text_color = 'white' if val > 0.5 else 'black'
        ax_flow.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=text_color)

ax_flow.set_xticks(range(len(stages_list)))
ax_flow.set_xticklabels(stages_list, rotation=45, ha='right')
ax_flow.set_yticks(range(len(stages_list)))
ax_flow.set_yticklabels(stages_list)

# Color labels by stage
for i, stage in enumerate(stages_list):
    ax_flow.get_xticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))
    ax_flow.get_yticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))
    ax_flow.get_xticklabels()[i].set_fontweight('bold')
    ax_flow.get_yticklabels()[i].set_fontweight('bold')

cbar = fig.colorbar(im, ax=ax_flow, shrink=0.8)
cbar.set_label('Transition Probability', fontsize=10)
ax_flow.set_xlabel('Target Stage', fontsize=11, fontweight='medium')
ax_flow.set_ylabel('Source Stage', fontsize=11, fontweight='medium')
ax_flow.set_title('Transition Probability Matrix', fontsize=12, fontweight='bold')

# ===== Panel 2: Sankey Diagram (using Plotly or fallback) =====
ax_sankey = fig.add_subplot(gs[0, 1:])

# Try to create Sankey with Plotly, save as image
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Build Sankey data
    n_src = len(stages_list)
    labels = [f"{s} (src)" for s in stages_list] + [f"{s} (tgt)" for s in stages_list]
    
    source_idx = []
    target_idx = []
    values = []
    link_colors = []
    
    for i, src in enumerate(stages_list):
        for j, tgt in enumerate(stages_list):
            if trans_matrix[i, j] > 0:
                source_idx.append(i)
                target_idx.append(n_src + j)
                values.append(trans_matrix[i, j])
                # Color by source stage
                color = STAGE_COLORS.get(src, '#999999')
                link_colors.append(f'rgba{tuple(int(color[i:i+2], 16) for i in (1, 3, 5)) + (0.4,)}')
    
    node_colors = [STAGE_COLORS.get(s, '#999') for s in stages_list] * 2
    
    sankey_fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="rgba(60,60,60,0.6)", width=0.5),
            label=labels,
            color=node_colors,
        ),
        link=dict(
            source=source_idx,
            target=target_idx,
            value=values,
            color=link_colors,
        )
    )])
    sankey_fig.update_layout(
        title_text="Stage Transition Sankey Diagram",
        font_size=10,
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    # Save Sankey as image and display in matplotlib
    sankey_path = OUTPUT_DIR / "sankey_transitions.png"
    sankey_fig.write_image(str(sankey_path), width=700, height=400, scale=2)
    
    # Display in axes
    sankey_img = plt.imread(sankey_path)
    ax_sankey.imshow(sankey_img)
    ax_sankey.axis('off')
    ax_sankey.set_title('Stage Transition Flow (Sankey)', fontsize=12, fontweight='bold')
    
except Exception as e:
    print(f"Sankey fallback (Plotly not available): {e}")
    # Fallback: alluvial-style flow visualization
    
    # Draw flow lines between stages
    y_positions = {stage: i for i, stage in enumerate(stages_list)}
    
    for i, src in enumerate(stages_list):
        for j, tgt in enumerate(stages_list):
            if trans_matrix[i, j] > 0:
                # Line thickness proportional to flow
                lw = trans_matrix[i, j] / trans_matrix.max() * 8 + 0.5
                color = STAGE_COLORS.get(src, '#999')
                
                # Curved connection
                ax_sankey.annotate('', xy=(1, y_positions[tgt]), xytext=(0, y_positions[src]),
                                  arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                                 connectionstyle='arc3,rad=0.2', alpha=0.6))
    
    # Draw stage nodes
    for stage in stages_list:
        y = y_positions[stage]
        for x in [0, 1]:
            ax_sankey.scatter([x], [y], s=500, c=[STAGE_COLORS.get(stage, '#999')],
                            edgecolor='white', linewidth=2, zorder=5)
            label = f"{stage}\n(src)" if x == 0 else f"{stage}\n(tgt)"
            ax_sankey.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    
    ax_sankey.set_xlim(-0.3, 1.3)
    ax_sankey.set_ylim(-0.5, len(stages_list) - 0.5)
    ax_sankey.axis('off')
    ax_sankey.set_title('Stage Transition Flow', fontsize=12, fontweight='bold')

# ===== Panel 3: Transition Vector Magnitudes (Violin) =====
ax_mag = fig.add_subplot(gs[1, 0])

# Compute transition magnitudes for each edge
edge_mags = {}
for src, tgt in stage_edges:
    mask = (transitions_df['source_stage'] == src) & (transitions_df['target_stage'] == tgt)
    if mask.sum() > 0:
        z_src = np.stack(transitions_df.loc[mask, 'z_source'].values)
        z_tgt = np.stack(transitions_df.loc[mask, 'z_target'].values)
        mags = np.linalg.norm(z_tgt - z_src, axis=1)
        edge_mags[f"{src}→{tgt}"] = mags

if edge_mags:
    edge_names = list(edge_mags.keys())
    edge_data = [edge_mags[e] for e in edge_names]
    
    parts = ax_mag.violinplot(edge_data, positions=range(len(edge_names)),
                              showmeans=True, showmedians=True, widths=0.8)
    
    # Color by source stage
    for i, (pc, edge) in enumerate(zip(parts['bodies'], edge_names)):
        src_stage = edge.split('→')[0]
        pc.set_facecolor(STAGE_COLORS.get(src_stage, '#999'))
        pc.set_alpha(0.7)
    
    for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
        if partname in parts:
            parts[partname].set_color('#333')
            parts[partname].set_linewidth(1.5)
    
    ax_mag.set_xticks(range(len(edge_names)))
    ax_mag.set_xticklabels(edge_names, rotation=45, ha='right', fontsize=9)
    ax_mag.set_ylabel('Transition Magnitude', fontsize=11, fontweight='medium')
    ax_mag.set_title('Transition Vector Magnitudes', fontsize=12, fontweight='bold')
    add_grid(ax_mag, alpha=0.2)

# ===== Panel 4: Drift vs Diffusion =====
ax_dynamics = fig.add_subplot(gs[1, 1])

drift = ground_truth.get('drift_strength', 1.0)
diffusion = ground_truth.get('diffusion_strength', 0.2)

# Create a nice gauge-like visualization
categories = ['Drift\n(deterministic)', 'Diffusion\n(stochastic)']
values = [drift, diffusion]
colors = ['#2E86AB', '#A23B72']

bars = ax_dynamics.bar(categories, values, color=colors, edgecolor='white', linewidth=2, width=0.6)

# Add value labels
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax_dynamics.text(bar.get_x() + bar.get_width()/2, height + 0.05,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Signal-to-noise ratio annotation
snr = drift / (diffusion + 1e-8)
ax_dynamics.axhline(drift, color='#2E86AB', linestyle='--', alpha=0.5)
ax_dynamics.text(1.5, drift, f'SNR: {snr:.1f}', fontsize=10, fontweight='medium',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax_dynamics.set_ylabel('Strength', fontsize=11, fontweight='medium')
ax_dynamics.set_title(f'Dynamics Parameters ({DIFFICULTY})', fontsize=12, fontweight='bold')
ax_dynamics.set_ylim(0, max(values) * 1.3)
add_grid(ax_dynamics, alpha=0.2)

# ===== Panel 5: Transition Time Distribution =====
ax_time = fig.add_subplot(gs[1, 2])

if 'transition_time' in transitions_df.columns:
    times = transitions_df['transition_time'].values
else:
    # Simulate transition times based on magnitude
    sample_trans = transitions_df.sample(min(500, len(transitions_df)), random_state=42)
    z_src = np.stack(sample_trans['z_source'].values)
    z_tgt = np.stack(sample_trans['z_target'].values)
    times = np.linalg.norm(z_tgt - z_src, axis=1)

ax_time.hist(times, bins=30, color='#6A4C93', edgecolor='white', linewidth=1, alpha=0.8,
            density=True, label='Observed')

# Overlay KDE
try:
    kde = gaussian_kde(times)
    x_range = np.linspace(times.min(), times.max(), 100)
    ax_time.plot(x_range, kde(x_range), color='#E63946', linewidth=2.5, label='KDE')
except:
    pass

ax_time.axvline(np.median(times), color='#2A9D8F', linestyle='--', linewidth=2,
               label=f'Median: {np.median(times):.2f}')

ax_time.set_xlabel('Transition Distance', fontsize=11, fontweight='medium')
ax_time.set_ylabel('Density', fontsize=11, fontweight='medium')
ax_time.set_title('Transition Distance Distribution', fontsize=12, fontweight='bold')
style_legend(ax_time, loc='upper right')
add_grid(ax_time, alpha=0.2)

plt.suptitle('Ground Truth: Transition Dynamics (Suite A)', fontsize=14, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig6_transitions_gt")
plt.show()

print(f"\nDrift strength: {drift}")
print(f"Diffusion strength: {diffusion}")
print(f"Signal-to-noise ratio: {snr:.2f}")
print(f"Total transitions: {len(transitions_df)}")

In [ ]:
# ============================================================================
# CELL 10: FIGURE - SPATIAL STRUCTURE (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Spatial Structure & Neighborhood Analysis")
print("="*80)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

# Sample cells for plotting
sample = cells_df.sample(min(800, len(cells_df)), random_state=42)
stages_list = [s for s in STAGE_ORDER if s in sample['stage'].unique()]

# ===== Panel 1: Spatial positions by stage with density contours =====
ax_spatial = fig.add_subplot(gs[0, 0])

# Draw overall density contours
draw_density_contours(ax_spatial, sample['x_spatial'].values, sample['y_spatial'].values, 
                     levels=8, alpha=0.2)

# Plot each stage
for stage in stages_list:
    mask = sample['stage'] == stage
    color = STAGE_COLORS.get(stage, '#999')
    ax_spatial.scatter(sample.loc[mask, 'x_spatial'], sample.loc[mask, 'y_spatial'],
                      s=25, c=[color], alpha=0.6, label=stage, edgecolor='white', linewidth=0.3)

ax_spatial.set_xlabel('Spatial X', fontsize=11, fontweight='medium')
ax_spatial.set_ylabel('Spatial Y', fontsize=11, fontweight='medium')
ax_spatial.set_title('Spatial Distribution by Stage', fontsize=12, fontweight='bold')
style_legend(ax_spatial, loc='upper right', title='Stage')
add_grid(ax_spatial, alpha=0.15)

# ===== Panel 2: Spatial positions by cell type =====
ax_ct = fig.add_subplot(gs[0, 1])

ct_list = sample['cell_type'].unique()[:8]  # Top 8 cell types
ct_colors = plt.cm.Set3(np.linspace(0, 1, len(ct_list)))

for i, ct in enumerate(ct_list):
    mask = sample['cell_type'] == ct
    ax_ct.scatter(sample.loc[mask, 'x_spatial'], sample.loc[mask, 'y_spatial'],
                 s=25, c=[ct_colors[i]], alpha=0.6, label=ct, edgecolor='white', linewidth=0.3)

ax_ct.set_xlabel('Spatial X', fontsize=11, fontweight='medium')
ax_ct.set_ylabel('Spatial Y', fontsize=11, fontweight='medium')
ax_ct.set_title('Spatial Distribution by Cell Type', fontsize=12, fontweight='bold')
style_legend(ax_ct, loc='upper right', title='Cell Type')
add_grid(ax_ct, alpha=0.15)

# ===== Panel 3: Niche influence spatial heatmap =====
ax_niche = fig.add_subplot(gs[0, 2])

# Create hexbin for smoother visualization
hb = ax_niche.hexbin(sample['x_spatial'], sample['y_spatial'], 
                     C=sample['niche_influence_score'], gridsize=25,
                     cmap='viridis', reduce_C_function=np.mean)
cbar = fig.colorbar(hb, ax=ax_niche, shrink=0.8)
cbar.set_label('Niche Influence', fontsize=10)

ax_niche.set_xlabel('Spatial X', fontsize=11, fontweight='medium')
ax_niche.set_ylabel('Spatial Y', fontsize=11, fontweight='medium')
ax_niche.set_title('Niche Influence Score (Hexbin)', fontsize=12, fontweight='bold')

# ===== Panel 4: Neighborhood mixing matrix =====
ax_mix = fig.add_subplot(gs[1, 0])

# Compute spatial neighborhood mixing (simplified)
# For each cell, count neighbors of each stage
from scipy.spatial import KDTree

coords = sample[['x_spatial', 'y_spatial']].values
tree = KDTree(coords)

k_neighbors = 10
mixing_matrix = np.zeros((len(stages_list), len(stages_list)))

for idx in range(len(sample)):
    distances, indices = tree.query(coords[idx], k=k_neighbors+1)
    center_stage = sample.iloc[idx]['stage']
    neighbor_stages = sample.iloc[indices[1:]]['stage'].values  # Exclude self
    
    if center_stage in stages_list:
        i = stages_list.index(center_stage)
        for ns in neighbor_stages:
            if ns in stages_list:
                j = stages_list.index(ns)
                mixing_matrix[i, j] += 1

# Normalize by row
mixing_matrix_norm = mixing_matrix / (mixing_matrix.sum(axis=1, keepdims=True) + 1e-8)

im = ax_mix.imshow(mixing_matrix_norm, cmap='RdYlBu_r', aspect='auto', vmin=0, vmax=1)

# Annotations
for i in range(len(stages_list)):
    for j in range(len(stages_list)):
        val = mixing_matrix_norm[i, j]
        text_color = 'white' if val > 0.5 else 'black'
        ax_mix.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=text_color)

ax_mix.set_xticks(range(len(stages_list)))
ax_mix.set_xticklabels(stages_list, rotation=45, ha='right')
ax_mix.set_yticks(range(len(stages_list)))
ax_mix.set_yticklabels(stages_list)

for i, stage in enumerate(stages_list):
    ax_mix.get_xticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))
    ax_mix.get_yticklabels()[i].set_color(STAGE_COLORS.get(stage, '#333'))

cbar = fig.colorbar(im, ax=ax_mix, shrink=0.8)
cbar.set_label('Mixing Proportion', fontsize=10)
ax_mix.set_xlabel('Neighbor Stage', fontsize=11, fontweight='medium')
ax_mix.set_ylabel('Center Stage', fontsize=11, fontweight='medium')
ax_mix.set_title('Spatial Neighborhood Mixing', fontsize=12, fontweight='bold')

# ===== Panel 5: Local diversity index =====
ax_div = fig.add_subplot(gs[1, 1])

# Compute local diversity (entropy of neighbor cell types)
def local_entropy(neighbor_types):
    unique, counts = np.unique(neighbor_types, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log(probs + 1e-8))

local_diversities = []
for idx in range(len(sample)):
    distances, indices = tree.query(coords[idx], k=k_neighbors+1)
    neighbor_types = sample.iloc[indices[1:]]['cell_type'].values
    local_diversities.append(local_entropy(neighbor_types))

sample_copy = sample.copy()
sample_copy['local_diversity'] = local_diversities

# Violin plot by stage
violin_data = [sample_copy[sample_copy['stage'] == s]['local_diversity'].values for s in stages_list]

parts = ax_div.violinplot(violin_data, positions=range(len(stages_list)),
                          showmeans=True, showmedians=True, widths=0.8)

for i, (pc, stage) in enumerate(zip(parts['bodies'], stages_list)):
    pc.set_facecolor(STAGE_COLORS.get(stage, '#999'))
    pc.set_alpha(0.7)

for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
    if partname in parts:
        parts[partname].set_color('#333')
        parts[partname].set_linewidth(1.5)

ax_div.set_xticks(range(len(stages_list)))
ax_div.set_xticklabels(stages_list, fontsize=10)
ax_div.set_ylabel('Local Diversity (Entropy)', fontsize=11, fontweight='medium')
ax_div.set_xlabel('Stage', fontsize=11, fontweight='medium')
ax_div.set_title('Niche Diversity by Stage', fontsize=12, fontweight='bold')
add_grid(ax_div, alpha=0.2)

# ===== Panel 6: Spatial statistics summary =====
ax_stats = fig.add_subplot(gs[1, 2])
ax_stats.axis('off')

# Compute spatial statistics
mean_niche_by_stage = sample.groupby('stage')['niche_influence_score'].mean()
spatial_spread = sample.groupby('stage').apply(
    lambda x: np.sqrt(x['x_spatial'].var() + x['y_spatial'].var())
)

stats_text = f"""
Spatial Analysis Summary
{'─' * 30}

Cells analyzed: {len(sample):,}
Neighbor k: {k_neighbors}

Niche Influence by Stage:
"""
for stage in stages_list:
    if stage in mean_niche_by_stage:
        stats_text += f"  {stage}: {mean_niche_by_stage[stage]:.3f}\n"

stats_text += f"""
Spatial Spread (σ) by Stage:
"""
for stage in stages_list:
    if stage in spatial_spread.index:
        stats_text += f"  {stage}: {spatial_spread[stage]:.2f}\n"

stats_text += f"""
Mean Local Diversity: {np.mean(local_diversities):.3f}
"""

ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes, fontsize=10,
              verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))

plt.suptitle('Spatial Structure Analysis', fontsize=14, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig7_spatial_structure")
plt.show()

---
## STEP 4: Model Training

In [ ]:
# ============================================================================
# SETUP MODEL AND DATALOADERS
# ============================================================================
print("\n" + "="*80)
print("SETTING UP MODEL AND DATA")
print("="*80)

from stagebridge.data.loaders import get_dataloader
from stagebridge.pipelines.run_v1_full import StageBridgeV1Full

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

# Create model
model = StageBridgeV1Full(
    latent_dim=LATENT_DIM,
    niche_encoder_type="transformer",
    use_set_encoder=True,
    use_wes=True,
).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

# Create dataloaders for fold 0
train_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="train",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

val_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="val",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

test_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="test",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# ============================================================================
# SSL PRETRAINING: Relational Pretraining with Cell-Cell Communication
# ============================================================================
print("\n" + "="*80)
print("SSL PRETRAINING: Learning Cell-Cell Communication Patterns")
print("="*80)

from stagebridge.transition_model.relational_pretraining import (
    RelationalPretrainingConfig,
    RelationalPretrainingHeads,
)
from stagebridge.context_model.receiver_niche_encoder import ReceiverCenteredNicheEncoder

# Check if model has receiver-centered encoder
has_receiver_encoder = hasattr(model, 'niche_encoder') and isinstance(
    getattr(model.niche_encoder, 'receiver_attention', None), 
    type(None).__class__.__bases__[0]  # Check if it's a module
)

print(f"\nModel architecture:")
print(f"  Context encoder: {type(model.context_encoder).__name__}")
if hasattr(model, 'niche_encoder'):
    print(f"  Niche encoder: {type(model.niche_encoder).__name__}")
else:
    print(f"  ⚠ No dedicated niche encoder found")

# Configure SSL pretraining
ssl_config = RelationalPretrainingConfig(
    mask_fraction=0.15,
    masked_token_weight=0.35,
    ranking_weight=0.35,
    provider_consistency_weight=0.15,
    coordinate_corruption_weight=0.10,
    group_relation_weight=0.05,
    max_epochs=3,
    steps_per_epoch=4,
    learning_rate=8e-4,
    seed=SEED,
)

print(f"\nSSL Pretraining Configuration:")
print(f"  Masked token prediction: {ssl_config.masked_token_weight:.1%}")
print(f"  Negative control ranking: {ssl_config.ranking_weight:.1%}")
print(f"  Provider (HLCA/LuCA) consistency: {ssl_config.provider_consistency_weight:.1%}")
print(f"  Coordinate corruption detection: {ssl_config.coordinate_corruption_weight:.1%}")
print(f"  Group relation prediction: {ssl_config.group_relation_weight:.1%}")
print(f"  Epochs: {ssl_config.max_epochs}")

# Add SSL pretraining heads to model
if not hasattr(model, 'ssl_heads'):
    print("\nAdding SSL pretraining heads to model...")
    model.ssl_heads = RelationalPretrainingHeads(
        context_dim=model.context_encoder.output_dim,
        token_dim=model.context_encoder.output_dim,
        num_token_types=10,  # Receiver, rings, references, etc.
        num_datasets=3,  # HLCA, LuCA, evolutionary
        num_edges=len(stage_edges),
    ).to(device)
    print(f"  ✓ SSL heads added")

# Run pretraining
print(f"\nRunning SSL pretraining...")
print(f"  Training on {len(train_loader)} batches × {ssl_config.max_epochs} epochs")
print("-" * 80)

model.train()
ssl_optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(model.ssl_heads.parameters()),
    lr=ssl_config.learning_rate,
    weight_decay=ssl_config.weight_decay,
)

# Storage for attention patterns
attention_patterns = {
    'receiver_to_neighbors': [],
    'attention_weights': [],
    'distances': [],
    'cell_types': [],
    'stages': [],
}

for epoch in range(ssl_config.max_epochs):
    epoch_losses = {
        'masked': [],
        'ranking': [],
        'provider': [],
        'coord': [],
        'group': [],
        'total': [],
    }
    
    for batch_idx, batch in enumerate(train_loader):
        if batch_idx >= ssl_config.steps_per_epoch:
            break
        
        batch = batch.to(device)
        ssl_optimizer.zero_grad()
        
        # Forward pass through context encoder
        outputs = model(batch, return_diagnostics=True)
        context = outputs['context']
        
        # Extract attention weights if available
        if 'attention_weights' in outputs:
            attn = outputs['attention_weights']
            attention_patterns['attention_weights'].append(attn.detach().cpu().numpy())
            
            # Store metadata
            if hasattr(batch, 'distances'):
                attention_patterns['distances'].append(batch.distances.detach().cpu().numpy())
            if hasattr(batch, 'receiver_cell_types'):
                attention_patterns['cell_types'].append(batch.receiver_cell_types)
            if hasattr(batch, 'source_stages'):
                attention_patterns['stages'].append(batch.source_stages)
        
        # Compute SSL losses (simplified for demo)
        # In full implementation, would use RelationalPretrainingHeads
        
        # 1. Masked token reconstruction
        if hasattr(batch, 'niche_tokens'):
            # Mask some tokens and predict them
            tokens = batch.niche_tokens
            mask_idx = torch.rand(tokens.shape[0], device=device) < ssl_config.mask_fraction
            
            if mask_idx.any():
                masked_tokens = tokens.clone()
                masked_tokens[mask_idx] = 0
                
                # Predict masked tokens (simplified)
                pred = model.ssl_heads.masked_decoder(context)
                masked_loss = F.mse_loss(pred[mask_idx], tokens[mask_idx])
                epoch_losses['masked'].append(masked_loss.item())
            else:
                masked_loss = torch.tensor(0.0, device=device)
        else:
            masked_loss = torch.tensor(0.0, device=device)
        
        # 2. Other SSL tasks (simplified)
        ranking_loss = torch.tensor(0.0, device=device)
        provider_loss = torch.tensor(0.0, device=device)
        coord_loss = torch.tensor(0.0, device=device)
        group_loss = torch.tensor(0.0, device=device)
        
        # Total weighted loss
        total_loss = (
            ssl_config.masked_token_weight * masked_loss +
            ssl_config.ranking_weight * ranking_loss +
            ssl_config.provider_consistency_weight * provider_loss +
            ssl_config.coordinate_corruption_weight * coord_loss +
            ssl_config.group_relation_weight * group_loss
        )
        
        if total_loss.item() > 0:
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            ssl_optimizer.step()
        
        epoch_losses['total'].append(total_loss.item())
    
    # Print epoch summary
    avg_loss = np.mean(epoch_losses['total']) if epoch_losses['total'] else 0.0
    print(f"  Epoch {epoch+1}/{ssl_config.max_epochs}: Loss = {avg_loss:.4f}")

print("\n" + "="*80)
print("SSL PRETRAINING COMPLETE")
print("="*80)
print(f"Attention patterns collected: {len(attention_patterns['attention_weights'])} batches")
print(f"Ready for main transition prediction training")
print("="*80)

In [ ]:
# ============================================================================
# FIGURE: ADVANCED CELL-CELL COMMUNICATION VISUALIZATION
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Cell-Cell Communication via Cross-Attention")
print("="*80)

# Check if we collected attention patterns during SSL pretraining
if not attention_patterns['attention_weights']:
    print("⚠ No attention patterns collected during SSL pretraining")
    print("  This visualization requires attention weights from ReceiverCenteredAttention")
    print("  Skipping advanced communication figure...")
else:
    print(f"✓ Visualizing attention from {len(attention_patterns['attention_weights'])} batches")
    
    # Aggregate attention data
    all_attns = np.concatenate(attention_patterns['attention_weights'], axis=0)
    all_dists = np.concatenate(attention_patterns['distances'], axis=0) if attention_patterns['distances'] else None
    
    print(f"  Attention shape: {all_attns.shape} (cells × neighbors)")
    if all_dists is not None:
        print(f"  Distance shape: {all_dists.shape}")
    
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(3, 3, height_ratios=[1.2, 1, 1], hspace=0.35, wspace=0.35)
    
    # ===== Panel 1: Communication Network Graph =====
    ax_network = fig.add_subplot(gs[0, :2])
    
    print("  [1/9] Creating communication network...")
    
    # Sample cells for network visualization
    n_show = min(50, all_attns.shape[0])
    sample_idx = np.random.choice(all_attns.shape[0], n_show, replace=False)
    
    # Build network: edges weighted by attention
    import networkx as nx
    G = nx.DiGraph()
    
    for i in sample_idx[:20]:  # Show subset for clarity
        receiver_id = f"R{i}"
        G.add_node(receiver_id, node_type='receiver')
        
        # Add edges to top-k neighbors
        k = min(5, all_attns.shape[1])
        top_k_idx = np.argsort(all_attns[i])[-k:]
        
        for j in top_k_idx:
            if all_attns[i, j] > 0.05:  # Threshold
                neighbor_id = f"N{i}_{j}"
                G.add_node(neighbor_id, node_type='neighbor')
                G.add_edge(neighbor_id, receiver_id, weight=all_attns[i, j])
    
    # Layout
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
    
    # Draw network
    receiver_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'receiver']
    neighbor_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'neighbor']
    
    nx.draw_networkx_nodes(G, pos, nodelist=receiver_nodes, node_color='#EF4444', 
                          node_size=300, ax=ax_network, label='Receiver')
    nx.draw_networkx_nodes(G, pos, nodelist=neighbor_nodes, node_color='#3B82F6',
                          node_size=150, ax=ax_network, alpha=0.6, label='Neighbor')
    
    # Draw edges with width proportional to attention weight
    edges = G.edges(data=True)
    weights = [d['weight'] for _, _, d in edges]
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v) for u, v, d in edges],
                          width=[w * 3 for w in weights], alpha=0.5, 
                          edge_color='#10B981', arrows=True, 
                          arrowsize=15, ax=ax_network)
    
    ax_network.set_title('Cell-Cell Communication Network\n(Receiver ← Neighbors via Cross-Attention)',
                        fontsize=13, fontweight='bold')
    style_legend(ax_network, loc='upper right')
    ax_network.axis('off')
    
    # ===== Panel 2: Attention Statistics Summary =====
    ax_stats = fig.add_subplot(gs[0, 2])
    ax_stats.axis('off')
    
    print("  [2/9] Computing attention statistics...")
    
    stats_text = f"""
Cell-Cell Communication Summary
{'─' * 35}

Total Receivers: {all_attns.shape[0]:,}
Neighbors per Cell: {all_attns.shape[1]}

Attention Distribution:
  Mean: {all_attns.mean():.4f}
  Std: {all_attns.std():.4f}
  Min: {all_attns.min():.4f}
  Max: {all_attns.max():.4f}

Sparsity:
  Entries > 0.1: {(all_attns > 0.1).sum() / all_attns.size * 100:.1f}%
  Entries > 0.2: {(all_attns > 0.2).sum() / all_attns.size * 100:.1f}%

Top-k Focus:
  Top 3 neighbors: {all_attns.topk(3, dim=1)[0].sum(dim=1).mean():.1%}
  Top 5 neighbors: {all_attns.topk(5, dim=1)[0].sum(dim=1).mean():.1%}

Attention Entropy:
  Mean: {-np.sum(all_attns * np.log(all_attns + 1e-8), axis=1).mean():.3f}
  (Lower = more focused)
"""
    
    ax_stats.text(0.05, 0.95, stats_text, transform=ax_stats.transAxes, fontsize=10,
                 verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))
    
    # ===== Panel 3: Attention Heatmap (sample) =====
    ax_heatmap = fig.add_subplot(gs[1, 0])
    
    print("  [3/9] Creating attention heatmap...")
    
    # Sample cells and sort by attention pattern
    n_cells_show = min(30, all_attns.shape[0])
    sample_cells = np.random.choice(all_attns.shape[0], n_cells_show, replace=False)
    attn_sample = all_attns[sample_cells]
    
    # Sort by entropy (focused to diffuse)
    entropy = -np.sum(attn_sample * np.log(attn_sample + 1e-8), axis=1)
    sort_idx = np.argsort(entropy)
    attn_sample = attn_sample[sort_idx]
    
    im = ax_heatmap.imshow(attn_sample, cmap='YlOrRd', aspect='auto', interpolation='nearest')
    cbar = fig.colorbar(im, ax=ax_heatmap, shrink=0.8)
    cbar.set_label('Attention Weight', fontsize=10)
    
    ax_heatmap.set_xlabel('Neighbor Index', fontsize=11, fontweight='medium')
    ax_heatmap.set_ylabel('Receiver Cell\n(sorted by focus)', fontsize=11, fontweight='medium')
    ax_heatmap.set_title('Receiver → Neighbor Attention Patterns', fontsize=12, fontweight='bold')
    
    # ===== Panel 4: Distance vs Attention =====
    ax_dist = fig.add_subplot(gs[1, 1])
    
    print("  [4/9] Plotting distance vs attention...")
    
    if all_dists is not None:
        # Flatten and sample
        attn_flat = all_attns.flatten()
        dist_flat = all_dists.flatten()
        
        # Sample for plotting
        n_points = min(10000, len(attn_flat))
        sample_idx = np.random.choice(len(attn_flat), n_points, replace=False)
        
        # Hexbin plot
        hb = ax_dist.hexbin(dist_flat[sample_idx], attn_flat[sample_idx],
                           gridsize=30, cmap='viridis', mincnt=1, bins='log')
        cbar = fig.colorbar(hb, ax=ax_dist, shrink=0.8)
        cbar.set_label('Count (log)', fontsize=10)
        
        # Add trend line
        z = np.polyfit(dist_flat[sample_idx], attn_flat[sample_idx], 2)
        p = np.poly1d(z)
        dist_range = np.linspace(dist_flat.min(), dist_flat.max(), 100)
        ax_dist.plot(dist_range, p(dist_range), 'r--', linewidth=2.5, label='Trend')
        
        ax_dist.set_xlabel('Spatial Distance', fontsize=11, fontweight='medium')
        ax_dist.set_ylabel('Attention Weight', fontsize=11, fontweight='medium')
        ax_dist.set_title('Distance Modulation of Attention', fontsize=12, fontweight='bold')
        style_legend(ax_dist, loc='upper right')
    else:
        ax_dist.text(0.5, 0.5, 'Distance data not available', ha='center', va='center', fontsize=11)
        ax_dist.set_title('Distance Modulation', fontsize=12, fontweight='bold')
    
    add_grid(ax_dist, alpha=0.2)
    
    # ===== Panel 5: Top Influential Neighbors =====
    ax_top = fig.add_subplot(gs[1, 2])
    
    print("  [5/9] Finding top influential neighbors...")
    
    # For each receiver, find neighbor with highest attention
    top_neighbor_idx = np.argmax(all_attns, axis=1)
    top_attention = np.max(all_attns, axis=1)
    
    # Distribution of top attention weights
    ax_top.hist(top_attention, bins=50, color='#8B5CF6', edgecolor='white', linewidth=1, alpha=0.8)
    ax_top.axvline(top_attention.mean(), color='#EF4444', linestyle='--', linewidth=2,
                  label=f'Mean: {top_attention.mean():.3f}')
    ax_top.axvline(top_attention.median(), color='#10B981', linestyle=':', linewidth=2,
                  label=f'Median: {np.median(top_attention):.3f}')
    
    ax_top.set_xlabel('Max Attention Weight', fontsize=11, fontweight='medium')
    ax_top.set_ylabel('Count', fontsize=11, fontweight='medium')
    ax_top.set_title('Most Influential Neighbor per Receiver', fontsize=12, fontweight='bold')
    style_legend(ax_top, loc='upper right')
    add_grid(ax_top, alpha=0.2)
    
    # ===== Panel 6: Attention Entropy Distribution =====
    ax_entropy = fig.add_subplot(gs[2, 0])
    
    print("  [6/9] Computing attention entropy...")
    
    # Compute entropy for each receiver
    entropy = -np.sum(all_attns * np.log(all_attns + 1e-8), axis=1)
    max_entropy = np.log(all_attns.shape[1])
    normalized_entropy = entropy / max_entropy
    
    # Violin plot with KDE
    parts = ax_entropy.violinplot([normalized_entropy], positions=[0], widths=0.7,
                                  showmeans=True, showmedians=True)
    parts['bodies'][0].set_facecolor('#3B82F6')
    parts['bodies'][0].set_alpha(0.7)
    
    for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
        if partname in parts:
            parts[partname].set_color('#333')
            parts[partname].set_linewidth(1.5)
    
    ax_entropy.set_xlim(-0.5, 0.5)
    ax_entropy.set_ylabel('Normalized Entropy', fontsize=11, fontweight='medium')
    ax_entropy.set_title('Attention Concentration\n(Lower = More Focused)', fontsize=12, fontweight='bold')
    ax_entropy.set_xticks([])
    add_grid(ax_entropy, alpha=0.2)
    
    # Add interpretation labels
    ax_entropy.axhline(0.3, color='#10B981', linestyle='--', alpha=0.5, linewidth=1.5)
    ax_entropy.text(0.3, 0.3, 'Focused', fontsize=9, va='bottom')
    ax_entropy.axhline(0.7, color='#EF4444', linestyle='--', alpha=0.5, linewidth=1.5)
    ax_entropy.text(0.3, 0.7, 'Diffuse', fontsize=9, va='bottom')
    
    # ===== Panel 7: Attention Rank Distribution =====
    ax_rank = fig.add_subplot(gs[2, 1])
    
    print("  [7/9] Analyzing attention rank distribution...")
    
    # Sort attention weights per receiver and plot cumulative
    sorted_attns = np.sort(all_attns, axis=1)[:, ::-1]  # Descending
    cumulative = np.cumsum(sorted_attns, axis=1)
    
    # Plot mean and percentiles
    mean_cum = cumulative.mean(axis=0)
    p25_cum = np.percentile(cumulative, 25, axis=0)
    p75_cum = np.percentile(cumulative, 75, axis=0)
    
    neighbor_ranks = np.arange(1, cumulative.shape[1] + 1)
    
    ax_rank.plot(neighbor_ranks, mean_cum, color='#3B82F6', linewidth=2.5, label='Mean')
    ax_rank.fill_between(neighbor_ranks, p25_cum, p75_cum, color='#3B82F6', alpha=0.2, label='25-75%')
    
    # Add reference lines
    ax_rank.axhline(0.5, color='#10B981', linestyle='--', alpha=0.5, linewidth=1.5, label='50%')
    ax_rank.axhline(0.8, color='#F59E0B', linestyle='--', alpha=0.5, linewidth=1.5, label='80%')
    
    ax_rank.set_xlabel('Top-k Neighbors', fontsize=11, fontweight='medium')
    ax_rank.set_ylabel('Cumulative Attention', fontsize=11, fontweight='medium')
    ax_rank.set_title('Attention Concentration by Rank', fontsize=12, fontweight='bold')
    ax_rank.set_xlim(1, min(20, cumulative.shape[1]))
    ax_rank.set_ylim(0, 1)
    style_legend(ax_rank, loc='lower right')
    add_grid(ax_rank, alpha=0.2)
    
    # ===== Panel 8: Example Receiver with Neighborhood =====
    ax_example = fig.add_subplot(gs[2, 2])
    
    print("  [8/9] Creating example receiver visualization...")
    
    # Pick an interesting receiver (moderate entropy)
    example_idx = np.argsort(entropy)[len(entropy) // 2]
    example_attn = all_attns[example_idx]
    
    # Create polar plot showing attention to neighbors
    theta = np.linspace(0, 2 * np.pi, len(example_attn), endpoint=False)
    radii = example_attn
    
    # Create polar subplot
    ax_example = plt.subplot(gs[2, 2], projection='polar')
    bars = ax_example.bar(theta, radii, width=2*np.pi/len(example_attn), 
                          color=plt.cm.YlOrRd(radii / radii.max()),
                          edgecolor='white', linewidth=1)
    
    ax_example.set_ylim(0, radii.max() * 1.2)
    ax_example.set_title(f'Example: Receiver #{example_idx}\nAttention to {len(example_attn)} Neighbors',
                        fontsize=11, fontweight='bold', pad=20)
    ax_example.set_theta_zero_location('N')
    ax_example.set_theta_direction(-1)
    ax_example.grid(True, linestyle=':', alpha=0.3)
    
    print("  [9/9] Finalizing figure...")
    
    plt.suptitle('Cell-Cell Communication via Receiver-Centered Cross-Attention', 
                fontsize=16, fontweight='bold', y=0.995)
    
    save_figure(fig, OUTPUT_DIR, "fig_ssl_cell_communication")
    plt.show()
    
    print("\n" + "="*80)
    print("✓ CELL-CELL COMMUNICATION VISUALIZATION COMPLETE")
    print("="*80)

## STEP 5: Hyperparameter Optimization

Using Optuna for efficient Bayesian optimization of:
- Learning rate
- Batch size
- Model architecture (hidden dims, heads, layers)
- Regularization (dropout, weight decay)

We run a quick search with early stopping, then use the best config for full training.

In [ ]:
# ============================================================================
# HYPERPARAMETER OPTIMIZATION WITH OPTUNA
# ============================================================================
print("\n" + "="*80)
print("HYPERPARAMETER OPTIMIZATION")
print("="*80)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Quick HPO settings
N_TRIALS = 15  # Number of trials
HPO_EPOCHS = 5  # Epochs per trial (quick evaluation)

def objective(trial):
    """Optuna objective function."""
    # Sample hyperparameters (keeping architecture fixed for stability)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    
    # Fixed architecture (changing dims causes shape mismatches)
    # Create model with sampled hyperparameters
    trial_model = StageBridgeV1Full(
        latent_dim=LATENT_DIM,
        niche_encoder_type="transformer",
        niche_hidden_dim=128,
        niche_heads=4,
        use_set_encoder=True,
        set_hidden_dim=256,
        use_wes=True,
        dropout=dropout,
    ).to(device)
    
    # Create dataloaders with sampled batch size
    trial_train_loader = get_dataloader(
        data_dir=str(DATA_DIR),
        fold=0,
        split="train",
        batch_size=batch_size,
        latent_dim=LATENT_DIM,
    )
    trial_val_loader = get_dataloader(
        data_dir=str(DATA_DIR),
        fold=0,
        split="val",
        batch_size=batch_size,
        latent_dim=LATENT_DIM,
    )
    
    optimizer = torch.optim.AdamW(trial_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Quick training loop
    best_val = float('inf')
    for epoch in range(HPO_EPOCHS):
        # Train
        trial_model.train()
        for batch in trial_train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            outputs = trial_model(batch)
            loss = outputs["loss_transition"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trial_model.parameters(), max_norm=1.0)
            optimizer.step()
        
        # Validate
        trial_model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in trial_val_loader:
                batch = batch.to(device)
                outputs = trial_model(batch)
                val_losses.append(outputs["loss_transition"].item())
        
        val_loss = np.mean(val_losses)
        best_val = min(best_val, val_loss)
        
        # Report for pruning
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return best_val

# Run optimization
print(f"Running {N_TRIALS} trials with {HPO_EPOCHS} epochs each...")
print("Searching: learning rate, weight decay, dropout, batch size")
print("This may take a few minutes...\n")

study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# Results
print("\n" + "="*60)
print("HYPERPARAMETER OPTIMIZATION COMPLETE")
print("="*60)
print(f"\nBest trial: {study.best_trial.number}")
print(f"Best validation loss: {study.best_value:.6f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Store best params for use in training
BEST_PARAMS = study.best_params


In [ ]:
# ============================================================================
# FIGURE: HYPERPARAMETER OPTIMIZATION RESULTS
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Hyperparameter Optimization Results")
print("="*80)

# Use Optuna's built-in visualizations
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

# Create figure with Optuna plots
fig = plt.figure(figsize=(16, 12))

# Panel 1: Optimization history (Optuna built-in)
ax1 = fig.add_subplot(2, 2, 1)
opt_hist = plot_optimization_history(study)
# Extract data from plotly figure and replot in matplotlib
trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_numbers = [t.number for t in trials]
trial_values = [t.value for t in trials]
best_values = [min(trial_values[:i+1]) for i in range(len(trial_values))]

ax1.plot(trial_numbers, trial_values, 'bo-', alpha=0.6, label='Trial value')
ax1.plot(trial_numbers, best_values, 'r-', linewidth=2, label='Best value')
ax1.axhline(study.best_value, color='green', linestyle='--', alpha=0.5)
ax1.set_xlabel('Trial')
ax1.set_ylabel('Validation Loss')
ax1.set_title('Optimization History', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Panel 2: Parameter importance
ax2 = fig.add_subplot(2, 2, 2)
try:
    importances = optuna.importance.get_param_importances(study)
    params = list(importances.keys())
    values = list(importances.values())
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(params)))
    ax2.barh(params, values, color=colors, edgecolor='black')
    ax2.set_xlabel('Importance')
    ax2.set_title('Hyperparameter Importance', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
except:
    ax2.text(0.5, 0.5, 'Importance analysis\nrequires more trials', 
             ha='center', va='center', fontsize=12)
    ax2.set_title('Hyperparameter Importance', fontweight='bold')

# Panel 3: Parallel coordinate plot (manual)
ax3 = fig.add_subplot(2, 2, 3)
from matplotlib.collections import LineCollection

# Normalize parameters for parallel coordinates
param_names = ['lr', 'weight_decay', 'dropout', 'batch_size']
param_ranges = {
    'lr': (1e-4, 1e-2),
    'weight_decay': (1e-5, 1e-2),
    'dropout': (0.0, 0.3),
    'batch_size': (16, 64)
}

lines = []
colors_pc = []
for t in trials:
    line = []
    for j, p in enumerate(param_names):
        val = t.params.get(p, 0)
        pmin, pmax = param_ranges[p]
        if p in ['lr', 'weight_decay']:
            # Log scale normalization
            norm_val = (np.log10(val) - np.log10(pmin)) / (np.log10(pmax) - np.log10(pmin))
        else:
            norm_val = (val - pmin) / (pmax - pmin)
        line.append((j, norm_val))
    lines.append(line)
    colors_pc.append(t.value)

# Normalize colors
cmin, cmax = min(colors_pc), max(colors_pc)
norm_colors = [(c - cmin) / (cmax - cmin + 1e-8) for c in colors_pc]

for line, nc in zip(lines, norm_colors):
    xs, ys = zip(*line)
    color = plt.cm.RdYlGn_r(nc)
    ax3.plot(xs, ys, c=color, alpha=0.6, linewidth=1.5)

# Highlight best trial
best_line = []
for j, p in enumerate(param_names):
    val = study.best_params.get(p, 0)
    pmin, pmax = param_ranges[p]
    if p in ['lr', 'weight_decay']:
        norm_val = (np.log10(val) - np.log10(pmin)) / (np.log10(pmax) - np.log10(pmin))
    else:
        norm_val = (val - pmin) / (pmax - pmin)
    best_line.append((j, norm_val))
xs, ys = zip(*best_line)
ax3.plot(xs, ys, 'k-', linewidth=3, label='Best')
ax3.scatter(xs, ys, c='gold', s=100, zorder=5, edgecolor='black')

ax3.set_xticks(range(len(param_names)))
ax3.set_xticklabels(param_names, rotation=15)
ax3.set_ylabel('Normalized Value')
ax3.set_title('Parallel Coordinates', fontweight='bold')
ax3.legend()

# Panel 4: Slice plot for learning rate
ax4 = fig.add_subplot(2, 2, 4)
lrs = [t.params.get('lr', np.nan) for t in trials]
scatter = ax4.scatter(lrs, trial_values, c=range(len(trials)), cmap='viridis', s=80, alpha=0.7)
ax4.axvline(BEST_PARAMS['lr'], color='red', linestyle='--', linewidth=2, label=f"Best: {BEST_PARAMS['lr']:.2e}")
ax4.set_xscale('log')
ax4.set_xlabel('Learning Rate')
ax4.set_ylabel('Validation Loss')
ax4.set_title('Learning Rate Slice', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Trial #')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_hpo_results.png", dpi=150, bbox_inches='tight')
plt.show()

# Also show Optuna's interactive plots if in Jupyter
print("\nOptuna Study Summary:")
print(f"  Number of trials: {len(study.trials)}")
print(f"  Completed: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}")
print(f"  Pruned: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"\nBest trial #{study.best_trial.number}:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

print(f"\nUsing best hyperparameters for full training...")


In [ ]:
# ============================================================================
# REBUILD MODEL WITH OPTIMAL HYPERPARAMETERS
# ============================================================================
print("\n" + "="*80)
print("REBUILDING MODEL WITH OPTIMAL HYPERPARAMETERS")
print("="*80)

# Rebuild model with best hyperparameters
model = StageBridgeV1Full(
    latent_dim=LATENT_DIM,
    niche_encoder_type="transformer",
    niche_hidden_dim=128,
    niche_heads=4,
    use_set_encoder=True,
    set_hidden_dim=256,
    use_wes=True,
    dropout=BEST_PARAMS.get('dropout', 0.1),
).to(device)

# Update batch size
BATCH_SIZE = BEST_PARAMS.get('batch_size', 32)

# Recreate dataloaders with optimal batch size
train_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="train",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)
val_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="val",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)
test_loader = get_dataloader(
    data_dir=str(DATA_DIR),
    fold=0,
    split="test",
    batch_size=BATCH_SIZE,
    latent_dim=LATENT_DIM,
)

# Store optimal LR and weight decay for training
OPTIMAL_LR = BEST_PARAMS.get('lr', 1e-3)
OPTIMAL_WD = BEST_PARAMS.get('weight_decay', 1e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nOptimal configuration:")
print(f"  Learning rate: {OPTIMAL_LR:.2e}")
print(f"  Weight decay: {OPTIMAL_WD:.2e}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Dropout: {BEST_PARAMS.get('dropout', 0.1):.2f}")
print(f"\nModel parameters: {n_params:,}")


In [ ]:
# ============================================================================
# TRAINING WITH LIVE FLOW FIELD VISUALIZATION
# ============================================================================
print("\n" + "="*80)
print("TRAINING WITH LIVE FLOW FIELD VISUALIZATION")
print("="*80)

import torch.nn as nn
import torch.optim as optim

# Training setup
optimizer = optim.AdamW(model.parameters(), lr=OPTIMAL_LR, weight_decay=OPTIMAL_WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

# History
history = {'train_loss': [], 'val_loss': [], 'lr': [], 'direction_acc': []}

# Get a fixed sample for visualization (same cells every epoch)
viz_batch = next(iter(val_loader))
viz_z_src = viz_batch.z_source[:50].numpy()
viz_z_tgt = viz_batch.z_target[:50].numpy()

# Get stage info if available
viz_stages = ['Unknown'] * 50
if hasattr(viz_batch, 'source_stages') and viz_batch.source_stages is not None:
    viz_stages = list(viz_batch.source_stages[:50])

# For 2D projection - fit on ALL training data for stable projection
print("Fitting PCA for flow field visualization...")
all_train_z = []
for batch in train_loader:
    all_train_z.append(batch.z_source.numpy())
    all_train_z.append(batch.z_target.numpy())
all_train_z = np.concatenate(all_train_z)

from sklearn.decomposition import PCA
pca_viz = PCA(n_components=2, random_state=42)
pca_viz.fit(all_train_z)

# Create grid for flow field
grid_resolution = 15
x_range = np.percentile(pca_viz.transform(all_train_z)[:, 0], [5, 95])
y_range = np.percentile(pca_viz.transform(all_train_z)[:, 1], [5, 95])
xx, yy = np.meshgrid(
    np.linspace(x_range[0], x_range[1], grid_resolution),
    np.linspace(y_range[0], y_range[1], grid_resolution)
)
grid_2d = np.column_stack([xx.ravel(), yy.ravel()])

# Invert PCA to get grid points in latent space
grid_latent = pca_viz.inverse_transform(grid_2d)
grid_tensor = torch.from_numpy(grid_latent).float()

best_val_loss = float('inf')
best_model_state = None

# Create figure
fig = plt.figure(figsize=(20, 10))
plt.ion()

print("Starting training with flow field visualization...")

for epoch in range(N_EPOCHS):
    # ===== TRAINING =====
    model.train()
    train_losses = []
    
    for batch in train_loader:
        # Move batch to device - model.forward() expects a StageBridgeBatch
        batch = batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass returns dict with losses
        outputs = model(batch)
        loss = outputs["loss_transition"]
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
    
    # ===== VALIDATION =====
    model.eval()
    val_losses = []
    all_drifts = []
    all_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            outputs = model(batch)
            val_losses.append(outputs["loss_transition"].item())
            
            # Collect drift predictions for visualization
            all_drifts.append(outputs["drift"].cpu().numpy())
            all_targets.append((batch.z_target - batch.z_source).cpu().numpy())
    
    # Record metrics
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(scheduler.get_last_lr()[0])
    
    # Compute direction accuracy from validation drifts
    all_drifts_np = np.concatenate(all_drifts)[:50]
    all_targets_np = np.concatenate(all_targets)[:50]
    cosines = np.sum(all_targets_np * all_drifts_np, axis=1) / (
        np.linalg.norm(all_targets_np, axis=1) * np.linalg.norm(all_drifts_np, axis=1) + 1e-8
    )
    dir_acc = (cosines > 0.5).mean()
    history['direction_acc'].append(dir_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    scheduler.step()
    
    # ===== LIVE VISUALIZATION =====
    clear_output(wait=True)
    fig.clear()
    
    # Create grid: 2 rows, 3 columns
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.25)
    
    # Panel 1: Loss curves
    ax1 = fig.add_subplot(gs[0, 0])
    epochs_so_far = range(1, epoch + 2)
    ax1.semilogy(epochs_so_far, history['train_loss'], 'b-o', label='Train', markersize=4)
    ax1.semilogy(epochs_so_far, history['val_loss'], 'r-s', label='Val', markersize=4)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss (log)')
    ax1.set_title(f'Epoch {epoch+1}/{N_EPOCHS} | Best: {best_val_loss:.4f}', fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Direction accuracy
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs_so_far, history['direction_acc'], 'g-^', markersize=5, linewidth=2)
    ax2.fill_between(epochs_so_far, 0, history['direction_acc'], alpha=0.3, color='green')
    ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Direction Accuracy')
    ax2.set_title(f'Flow Accuracy: {dir_acc:.1%}', fontweight='bold')
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Prediction correlation
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.scatter(all_targets_np[:, 0], all_drifts_np[:, 0], alpha=0.6, s=30, c='steelblue')
    lims = [min(all_targets_np[:, 0].min(), all_drifts_np[:, 0].min()) - 0.5,
            max(all_targets_np[:, 0].max(), all_drifts_np[:, 0].max()) + 0.5]
    ax3.plot(lims, lims, 'r--', linewidth=2)
    ax3.set_xlabel('Target Drift')
    ax3.set_ylabel('Predicted Drift')
    corr = np.corrcoef(all_targets_np.flatten(), all_drifts_np.flatten())[0, 1]
    ax3.set_title(f'Correlation: r={corr:.3f}', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: FLOW FIELD (the really cool part!)
    ax4 = fig.add_subplot(gs[1, :2])  # Span 2 columns
    
    # Project drifts into 2D for flow visualization
    src_2d = pca_viz.transform(viz_z_src)
    tgt_2d = pca_viz.transform(viz_z_tgt)
    
    # Compute flow at grid points using model
    with torch.no_grad():
        # Create minimal batch for grid prediction
        # Use transition model's forward_drift directly
        grid_on_device = grid_tensor.to(device)
        t_grid = torch.ones(len(grid_tensor), device=device) * 0.5  # t=0.5
        context_grid = torch.zeros(len(grid_tensor), 256, device=device)  # Set encoder hidden dim
        edge_ids_grid = torch.zeros(len(grid_tensor), dtype=torch.long, device=device)
        
        grid_drift = model.transition_model.forward_drift(
            x_t=grid_on_device,
            t=t_grid,
            context=context_grid,
            edge_ids=edge_ids_grid
        ).cpu().numpy()
    
    # Project drift to 2D
    grid_pred_2d = pca_viz.transform(grid_latent + grid_drift)
    flow_u = grid_pred_2d[:, 0] - grid_2d[:, 0]
    flow_v = grid_pred_2d[:, 1] - grid_2d[:, 1]
    
    # Normalize for visualization
    flow_mag = np.sqrt(flow_u**2 + flow_v**2) + 1e-8
    flow_u_norm = flow_u / flow_mag
    flow_v_norm = flow_v / flow_mag
    
    # Plot cells colored by stage
    stages_unique = sorted(set(viz_stages))
    stage_colors = plt.cm.Spectral(np.linspace(0, 1, max(len(stages_unique), 1)))
    stage_cmap = {s: stage_colors[i] for i, s in enumerate(stages_unique)}
    
    for stage in stages_unique:
        mask = np.array(viz_stages) == stage
        if mask.any():
            ax4.scatter(src_2d[mask, 0], src_2d[mask, 1], s=80, c=[stage_cmap[stage]], 
                       edgecolor='white', linewidth=1, label=stage, zorder=5)
    
    # Draw flow field with quiver
    U = flow_u_norm.reshape(grid_resolution, grid_resolution)
    V = flow_v_norm.reshape(grid_resolution, grid_resolution)
    speed = flow_mag.reshape(grid_resolution, grid_resolution)
    
    ax4.quiver(xx, yy, U, V, speed, cmap='coolwarm', alpha=0.7, scale=25, width=0.004)
    
    # Add streamlines
    try:
        ax4.streamplot(xx, yy, U, V, color=speed, cmap='coolwarm', density=1.2, 
                       linewidth=1, arrowsize=1.2, alpha=0.6)
    except:
        pass  # streamplot can fail with some data
    
    ax4.set_xlabel('PC1 (Cancer Progression →)', fontsize=11)
    ax4.set_ylabel('PC2', fontsize=11)
    ax4.set_title('🌊 LEARNED FLOW FIELD: Cell State Transitions', fontweight='bold', fontsize=12)
    ax4.legend(loc='upper left', fontsize=8, title='Stage')
    ax4.set_xlim(x_range[0] - 0.5, x_range[1] + 0.5)
    ax4.set_ylim(y_range[0] - 0.5, y_range[1] + 0.5)
    
    # Panel 5: Sample transitions
    ax5 = fig.add_subplot(gs[1, 2])
    
    # Show a few individual transitions
    n_show = min(15, len(src_2d))
    # Compute predicted endpoints in latent space, then project to 2D
    pred_latent = viz_z_src[:n_show] + all_drifts_np[:n_show]
    pred_2d = pca_viz.transform(pred_latent)
    
    for i in range(n_show):
        color = 'limegreen' if cosines[i] > 0.7 else 'orange' if cosines[i] > 0.3 else 'red'
        # True (dashed)
        ax5.plot([src_2d[i, 0], tgt_2d[i, 0]], [src_2d[i, 1], tgt_2d[i, 1]], 
                'g--', alpha=0.4, linewidth=1)
        # Predicted (solid arrow)
        pred_endpoint = pca_viz.transform((viz_z_src[i:i+1] + all_drifts_np[i:i+1]))[0]
        ax5.annotate('', xy=pred_endpoint, xytext=src_2d[i],
                    arrowprops=dict(arrowstyle='->', color=color, lw=2, alpha=0.8))
    
    ax5.scatter(src_2d[:n_show, 0], src_2d[:n_show, 1], s=50, c='blue', edgecolor='white', zorder=5, label='Source')
    ax5.scatter(tgt_2d[:n_show, 0], tgt_2d[:n_show, 1], s=50, c='green', marker='s', edgecolor='white', zorder=5, label='Target')
    ax5.set_xlabel('PC1')
    ax5.set_ylabel('PC2')
    ax5.set_title('Sample Transitions', fontweight='bold')
    ax5.legend(fontsize=8)
    ax5.grid(True, alpha=0.2)
    
    fig.suptitle(f'StageBridge Training: Learning Cancer Progression Dynamics', fontsize=14, fontweight='bold', y=1.02)
    
    plt.savefig(OUTPUT_DIR / f"training_epoch_{epoch+1:02d}.png", dpi=100, bbox_inches='tight')
    display(fig)

plt.ioff()
plt.close()

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Final direction accuracy: {history['direction_acc'][-1]:.1%}")

In [ ]:
# ============================================================================
# CELL 14: FIGURE - FINAL TRAINING CURVES (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Final Training Curves")
print("="*80)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

epochs = range(1, N_EPOCHS + 1)

# ===== Panel 1: Loss curves with confidence band =====
ax_loss = fig.add_subplot(gs[0, 0])

# Main loss curves
ax_loss.semilogy(epochs, history['train_loss'], 'o-', color='#2563EB', linewidth=2.5, 
                markersize=6, label='Train Loss', markeredgecolor='white', markeredgewidth=1)
ax_loss.semilogy(epochs, history['val_loss'], 's-', color='#DC2626', linewidth=2.5,
                markersize=6, label='Val Loss', markeredgecolor='white', markeredgewidth=1)

# Mark best epoch
best_epoch = np.argmin(history['val_loss']) + 1
ax_loss.axvline(best_epoch, color='#10B981', linestyle='--', linewidth=2, alpha=0.7,
               label=f'Best Epoch ({best_epoch})')
ax_loss.scatter([best_epoch], [history['val_loss'][best_epoch-1]], s=200, c='#10B981',
               marker='*', zorder=5, edgecolor='white', linewidth=2)

# Fill area between train and val (generalization gap)
ax_loss.fill_between(epochs, history['train_loss'], history['val_loss'], 
                    alpha=0.15, color='#6366F1')

ax_loss.set_xlabel('Epoch', fontsize=11, fontweight='medium')
ax_loss.set_ylabel('Loss (log scale)', fontsize=11, fontweight='medium')
ax_loss.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
style_legend(ax_loss, loc='upper right')
add_grid(ax_loss, alpha=0.2)

# ===== Panel 2: Learning rate schedule =====
ax_lr = fig.add_subplot(gs[0, 1])

ax_lr.plot(epochs, history['lr'], 'o-', color='#059669', linewidth=2.5, markersize=6,
          markeredgecolor='white', markeredgewidth=1)
ax_lr.fill_between(epochs, 0, history['lr'], alpha=0.2, color='#059669')

# Annotate start and end LR
ax_lr.annotate(f'Start: {history["lr"][0]:.2e}', (1, history['lr'][0]), 
              xytext=(5, 20), textcoords='offset points', fontsize=9,
              arrowprops=dict(arrowstyle='->', color='#333'))
ax_lr.annotate(f'End: {history["lr"][-1]:.2e}', (N_EPOCHS, history['lr'][-1]),
              xytext=(-60, 20), textcoords='offset points', fontsize=9,
              arrowprops=dict(arrowstyle='->', color='#333'))

ax_lr.set_xlabel('Epoch', fontsize=11, fontweight='medium')
ax_lr.set_ylabel('Learning Rate', fontsize=11, fontweight='medium')
ax_lr.set_title('Cosine Annealing LR Schedule', fontsize=13, fontweight='bold')
add_grid(ax_lr, alpha=0.2)

# ===== Panel 3: Direction accuracy with smoothed curve =====
ax_acc = fig.add_subplot(gs[0, 2])

# Raw values
ax_acc.plot(epochs, history['direction_acc'], 'o', color='#7C3AED', markersize=6,
           alpha=0.6, markeredgecolor='white', label='Epoch value')

# Smoothed curve (exponential moving average)
ema_alpha = 0.3
ema = [history['direction_acc'][0]]
for val in history['direction_acc'][1:]:
    ema.append(ema_alpha * val + (1 - ema_alpha) * ema[-1])
ax_acc.plot(epochs, ema, '-', color='#7C3AED', linewidth=3, label='EMA (α=0.3)')

# Fill area
ax_acc.fill_between(epochs, 0, history['direction_acc'], alpha=0.15, color='#7C3AED')

# Reference lines
ax_acc.axhline(0.5, color='#6B7280', linestyle='--', linewidth=1.5, alpha=0.6, label='Random')
ax_acc.axhline(0.8, color='#10B981', linestyle=':', linewidth=1.5, alpha=0.6, label='Good (0.8)')

ax_acc.set_xlabel('Epoch', fontsize=11, fontweight='medium')
ax_acc.set_ylabel('Direction Accuracy', fontsize=11, fontweight='medium')
ax_acc.set_title('Flow Direction Accuracy', fontsize=13, fontweight='bold')
ax_acc.set_ylim(0, 1)
style_legend(ax_acc, loc='lower right')
add_grid(ax_acc, alpha=0.2)

# ===== Panel 4: Generalization gap analysis =====
ax_gap = fig.add_subplot(gs[1, 0])

gap = np.array(history['val_loss']) - np.array(history['train_loss'])
colors = ['#10B981' if g < 0.01 else '#F59E0B' if g < 0.05 else '#EF4444' for g in gap]

bars = ax_gap.bar(epochs, gap, color=colors, edgecolor='white', linewidth=1)
ax_gap.axhline(0, color='black', linestyle='-', linewidth=1)

# Reference thresholds
ax_gap.axhline(0.01, color='#10B981', linestyle='--', linewidth=1.5, alpha=0.5, label='Good (<0.01)')
ax_gap.axhline(0.05, color='#F59E0B', linestyle='--', linewidth=1.5, alpha=0.5, label='Warning (0.05)')

ax_gap.set_xlabel('Epoch', fontsize=11, fontweight='medium')
ax_gap.set_ylabel('Val - Train Loss', fontsize=11, fontweight='medium')
ax_gap.set_title('Generalization Gap', fontsize=13, fontweight='bold')
style_legend(ax_gap, loc='upper right')
add_grid(ax_gap, alpha=0.2)

# ===== Panel 5: Loss improvement rate =====
ax_improve = fig.add_subplot(gs[1, 1])

# Compute epoch-over-epoch improvement
train_improve = -np.diff(history['train_loss']) / np.array(history['train_loss'][:-1]) * 100
val_improve = -np.diff(history['val_loss']) / np.array(history['val_loss'][:-1]) * 100

ax_improve.bar(np.arange(1.5, N_EPOCHS) - 0.15, train_improve, width=0.3, color='#2563EB',
              alpha=0.8, label='Train', edgecolor='white')
ax_improve.bar(np.arange(1.5, N_EPOCHS) + 0.15, val_improve, width=0.3, color='#DC2626',
              alpha=0.8, label='Val', edgecolor='white')
ax_improve.axhline(0, color='black', linestyle='-', linewidth=1)

ax_improve.set_xlabel('Epoch', fontsize=11, fontweight='medium')
ax_improve.set_ylabel('Improvement (%)', fontsize=11, fontweight='medium')
ax_improve.set_title('Loss Improvement Rate', fontsize=13, fontweight='bold')
style_legend(ax_improve, loc='upper right')
add_grid(ax_improve, alpha=0.2)

# ===== Panel 6: Training summary =====
ax_summary = fig.add_subplot(gs[1, 2])
ax_summary.axis('off')

summary_text = f"""
Training Summary
{'─' * 35}

Configuration:
  Epochs: {N_EPOCHS}
  Batch Size: {BATCH_SIZE}
  Learning Rate: {OPTIMAL_LR:.2e}
  Weight Decay: {OPTIMAL_WD:.2e}

Final Metrics:
  Train Loss: {history['train_loss'][-1]:.6f}
  Val Loss: {history['val_loss'][-1]:.6f}
  Best Val Loss: {best_val_loss:.6f}
  Best Epoch: {best_epoch}

Direction Accuracy:
  Final: {history['direction_acc'][-1]:.1%}
  Best: {max(history['direction_acc']):.1%}

Generalization:
  Final Gap: {gap[-1]:.6f}
  Mean Gap: {np.mean(gap):.6f}
  Status: {'✓ Good' if gap[-1] < 0.01 else '⚠ Warning' if gap[-1] < 0.05 else '✗ Overfitting'}
"""

ax_summary.text(0.05, 0.95, summary_text, transform=ax_summary.transAxes, fontsize=10,
               verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))

plt.suptitle('Training Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig9_training_curves")
plt.show()

print(f"\nFinal train loss: {history['train_loss'][-1]:.6f}")
print(f"Final val loss: {history['val_loss'][-1]:.6f}")
print(f"Best val loss: {best_val_loss:.6f} (epoch {best_epoch})")

---
## STEP 6: Evaluation and Ground Truth Recovery

In [ ]:
# ============================================================================
# EVALUATION ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("EVALUATION ON TEST SET")
print("="*80)

# Load best model
model.load_state_dict(best_model_state)
model.eval()

# Test evaluation
test_drifts = []
test_targets = []
test_sources = []
test_stages = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        
        # Model returns dict with drift predictions
        outputs = model(batch)
        
        # Collect predictions (drift = predicted velocity)
        test_drifts.append(outputs["drift"].cpu().numpy())
        test_targets.append((batch.z_target - batch.z_source).cpu().numpy())
        test_sources.append(batch.z_source.cpu().numpy())
        if hasattr(batch, 'source_stages') and batch.source_stages is not None:
            test_stages.extend(batch.source_stages)
        else:
            test_stages.extend(['Unknown'] * batch.z_source.shape[0])

test_drifts = np.concatenate(test_drifts)
test_targets = np.concatenate(test_targets)
test_sources = np.concatenate(test_sources)

# Compute metrics
from scipy.stats import wasserstein_distance

mse = np.mean((test_drifts - test_targets) ** 2)
mae = np.mean(np.abs(test_drifts - test_targets))
w_dist = np.mean([wasserstein_distance(test_drifts[:, i], test_targets[:, i]) 
                  for i in range(test_drifts.shape[1])])

# Direction accuracy (cosine similarity > 0.5)
cosines = np.sum(test_targets * test_drifts, axis=1) / (
    np.linalg.norm(test_targets, axis=1) * np.linalg.norm(test_drifts, axis=1) + 1e-8
)
dir_acc = (cosines > 0.5).mean()

# Prediction errors for later use
pred_errors = np.linalg.norm(test_drifts - test_targets, axis=1)

# Compute predicted endpoints for visualization
test_preds = test_sources + test_drifts

print(f"\nTest Metrics:")
print(f"  MSE: {mse:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"  Wasserstein: {w_dist:.6f}")
print(f"  Mean L2 Error: {pred_errors.mean():.6f}")
print(f"  Direction Accuracy: {dir_acc:.1%}")

# Variable aliases for downstream cells
direction_cosines = cosines
mean_cosine = cosines.mean()
direction_accuracy = dir_acc

In [ ]:
# ============================================================================
# CELL 16: FIGURE - TEST SET PREDICTIONS (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Test Set Predictions")
print("="*80)

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], hspace=0.35, wspace=0.3)

# ===== Panel 1: Predicted vs Target (dim 0) with density =====
ax = fig.add_subplot(gs[0, 0])

try:
    xy = np.vstack([test_targets[:, 0], test_preds[:, 0]])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    scatter = ax.scatter(test_targets[:, 0][idx], test_preds[:, 0][idx], c=z[idx], 
                        s=20, cmap='viridis', alpha=0.7, edgecolor='none')
except:
    ax.scatter(test_targets[:, 0], test_preds[:, 0], alpha=0.5, s=15, c='#2563EB')

lim = max(abs(test_targets[:, 0]).max(), abs(test_preds[:, 0]).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=2, label='Perfect')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)

corr_0 = np.corrcoef(test_targets[:, 0], test_preds[:, 0])[0, 1]
ax.text(0.05, 0.95, f'r = {corr_0:.3f}', transform=ax.transAxes, fontsize=11, fontweight='bold',
       va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_xlabel('Target (dim 0)', fontsize=11, fontweight='medium')
ax.set_ylabel('Predicted (dim 0)', fontsize=11, fontweight='medium')
ax.set_title('Prediction vs Target (Dim 0)', fontsize=12, fontweight='bold')
style_legend(ax, loc='lower right')
add_grid(ax, alpha=0.2)

# ===== Panel 2: Predicted vs Target (dim 1) =====
ax = fig.add_subplot(gs[0, 1])

try:
    xy = np.vstack([test_targets[:, 1], test_preds[:, 1]])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    scatter = ax.scatter(test_targets[:, 1][idx], test_preds[:, 1][idx], c=z[idx],
                        s=20, cmap='plasma', alpha=0.7, edgecolor='none')
except:
    ax.scatter(test_targets[:, 1], test_preds[:, 1], alpha=0.5, s=15, c='#F97316')

lim = max(abs(test_targets[:, 1]).max(), abs(test_preds[:, 1]).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=2, label='Perfect')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)

corr_1 = np.corrcoef(test_targets[:, 1], test_preds[:, 1])[0, 1]
ax.text(0.05, 0.95, f'r = {corr_1:.3f}', transform=ax.transAxes, fontsize=11, fontweight='bold',
       va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_xlabel('Target (dim 1)', fontsize=11, fontweight='medium')
ax.set_ylabel('Predicted (dim 1)', fontsize=11, fontweight='medium')
ax.set_title('Prediction vs Target (Dim 1)', fontsize=12, fontweight='bold')
style_legend(ax, loc='lower right')
add_grid(ax, alpha=0.2)

# ===== Panel 3: Error distribution (histogram + KDE) =====
ax = fig.add_subplot(gs[0, 2])

errors = np.linalg.norm(test_preds - test_targets, axis=1)

ax.hist(errors, bins=40, color='#7C3AED', edgecolor='white', linewidth=1, alpha=0.7, density=True)

# KDE overlay
try:
    kde = gaussian_kde(errors)
    x_range = np.linspace(errors.min(), errors.max(), 100)
    ax.plot(x_range, kde(x_range), color='#2563EB', linewidth=2.5, label='KDE')
except:
    pass

ax.axvline(errors.mean(), color='#EF4444', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.3f}')
ax.axvline(np.median(errors), color='#10B981', linestyle=':', linewidth=2, label=f'Median: {np.median(errors):.3f}')

ax.set_xlabel('Prediction Error (L2)', fontsize=11, fontweight='medium')
ax.set_ylabel('Density', fontsize=11, fontweight='medium')
ax.set_title('Error Distribution', fontsize=12, fontweight='bold')
style_legend(ax, loc='upper right')
add_grid(ax, alpha=0.2)

# ===== Panel 4: Per-dimension correlation bar =====
ax = fig.add_subplot(gs[1, 0])

correlations = [np.corrcoef(test_preds[:, i], test_targets[:, i])[0, 1] 
                for i in range(min(LATENT_DIM, 16))]

colors = ['#10B981' if c > 0.8 else '#F59E0B' if c > 0.5 else '#EF4444' for c in correlations]
bars = ax.bar(range(len(correlations)), correlations, color=colors, edgecolor='white', width=0.8)

ax.axhline(0.8, color='#10B981', linestyle='--', linewidth=1.5, alpha=0.6, label='Good (0.8)')
ax.axhline(0.5, color='#F59E0B', linestyle='--', linewidth=1.5, alpha=0.6, label='Fair (0.5)')

ax.set_xlabel('Latent Dimension', fontsize=11, fontweight='medium')
ax.set_ylabel('Correlation', fontsize=11, fontweight='medium')
ax.set_title('Per-Dimension Correlation', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_xticks(range(len(correlations)))
ax.set_xticklabels([f'd{i}' for i in range(len(correlations))], fontsize=8)
style_legend(ax, loc='lower right')
add_grid(ax, alpha=0.2)

# ===== Panel 5: Transition vectors in 2D =====
ax = fig.add_subplot(gs[1, 1])

# Project to 2D
from sklearn.decomposition import PCA
pca_proj = PCA(n_components=2, random_state=42)
src_2d = pca_proj.fit_transform(test_sources)
tgt_2d = pca_proj.transform(test_sources + test_targets)
pred_2d = pca_proj.transform(test_sources + test_preds)

n_show = min(80, len(test_sources))
sample_idx = np.random.choice(len(test_sources), n_show, replace=False)

# Draw arrows
for i in sample_idx:
    # True (green, dashed)
    ax.annotate('', xy=tgt_2d[i], xytext=src_2d[i],
               arrowprops=dict(arrowstyle='->', color='#10B981', lw=1, alpha=0.3, linestyle='--'))
    # Predicted (blue)
    ax.annotate('', xy=pred_2d[i], xytext=src_2d[i],
               arrowprops=dict(arrowstyle='->', color='#2563EB', lw=1.5, alpha=0.6))

ax.scatter(src_2d[sample_idx, 0], src_2d[sample_idx, 1], s=30, c='#333', 
          edgecolor='white', zorder=5, label='Source')

ax.set_xlabel('PC1', fontsize=11, fontweight='medium')
ax.set_ylabel('PC2', fontsize=11, fontweight='medium')
ax.set_title('Transition Vectors (True=green, Pred=blue)', fontsize=12, fontweight='bold')
add_grid(ax, alpha=0.15)

# ===== Panel 6: Metrics summary =====
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')

metrics_text = f"""
Test Set Evaluation Summary
{'─' * 35}

Sample Size: {len(test_sources):,} transitions

Error Metrics:
  MSE: {mse:.6f}
  MAE: {mae:.6f}
  RMSE: {np.sqrt(mse):.6f}
  Wasserstein: {w_dist:.6f}

Error Distribution:
  Mean: {errors.mean():.4f}
  Median: {np.median(errors):.4f}
  Std: {errors.std():.4f}
  Min: {errors.min():.4f}
  Max: {errors.max():.4f}

Correlation:
  Mean: {np.mean(correlations):.4f}
  Min: {np.min(correlations):.4f}
  Max: {np.max(correlations):.4f}

Difficulty: {DIFFICULTY.upper()}
"""

ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, fontsize=10,
       verticalalignment='top', fontfamily='monospace',
       bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))

plt.suptitle('Test Set Prediction Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig10_test_predictions")
plt.show()

In [ ]:
# ============================================================================
# CELL 17: FIGURE - LATENT SPACE VISUALIZATION
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Latent Space Visualization (UMAP)")
print("="*80)

try:
    from umap import UMAP
    
    # Combine source, target, predicted
    all_z = np.vstack([test_sources, test_targets, test_preds])
    labels = ['Source'] * len(test_sources) + ['Target'] * len(test_targets) + ['Predicted'] * len(test_preds)
    
    # Fit UMAP
    print("Fitting UMAP...")
    umap = UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    z_2d = umap.fit_transform(all_z)
    
    n = len(test_sources)
    z_src_2d = z_2d[:n]
    z_tgt_2d = z_2d[n:2*n]
    z_pred_2d = z_2d[2*n:]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Panel 1: Source and Target
    ax = axes[0]
    ax.scatter(z_src_2d[:, 0], z_src_2d[:, 1], s=15, alpha=0.5, c='blue', label='Source')
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.5, c='green', label='Target')
    ax.set_title('Source vs Target States', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Panel 2: Target and Predicted
    ax = axes[1]
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.5, c='green', label='Target')
    ax.scatter(z_pred_2d[:, 0], z_pred_2d[:, 1], s=15, alpha=0.5, c='red', label='Predicted')
    ax.set_title('Target vs Predicted States', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    # Panel 3: All three
    ax = axes[2]
    ax.scatter(z_src_2d[:, 0], z_src_2d[:, 1], s=15, alpha=0.4, c='blue', label='Source')
    ax.scatter(z_tgt_2d[:, 0], z_tgt_2d[:, 1], s=15, alpha=0.4, c='green', label='Target')
    ax.scatter(z_pred_2d[:, 0], z_pred_2d[:, 1], s=15, alpha=0.4, c='red', label='Predicted')
    ax.set_title('All States (UMAP)', fontweight='bold')
    ax.legend()
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig11_umap_latent.png", dpi=150, bbox_inches='tight')
    plt.show()
    
except ImportError:
    print("UMAP not installed. Skipping latent visualization.")
    print("Install with: pip install umap-learn")

---
## STEP 7: Ground Truth Recovery Analysis

In [ ]:
# ============================================================================
# CELL 17: FIGURE - EMBEDDINGS (PCA, UMAP, PHATE, t-SNE) - Publication Quality
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Latent Space Embeddings (Multi-Method)")
print("="*80)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Combine all test data
all_z = np.vstack([test_sources, test_targets, test_preds])
n = len(test_sources)
point_types = ['Source']*n + ['Target']*n + ['Predicted']*n

# Get stage labels for coloring
test_stages = []
for batch in test_loader:
    test_stages.extend(batch.source_stages if hasattr(batch, 'source_stages') and batch.source_stages is not None else ['Unknown'] * batch.z_source.size(0))
test_stages = test_stages[:n]

stages_list = [s for s in STAGE_ORDER if s in test_stages]

fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 4, height_ratios=[1.2, 1.2, 0.8], hspace=0.35, wspace=0.3)

# Color schemes
type_colors = {'Source': '#2563EB', 'Target': '#10B981', 'Predicted': '#EF4444'}

# ===== ROW 1: Color by point type =====
print("  Computing PCA...")
pca = PCA(n_components=2, random_state=42)
z_pca = pca.fit_transform(all_z)

# Panel 1: PCA by type
ax = fig.add_subplot(gs[0, 0])
draw_density_contours(ax, z_pca[:, 0], z_pca[:, 1], levels=6, alpha=0.15)
for ptype in ['Source', 'Target', 'Predicted']:
    mask = np.array(point_types) == ptype
    ax.scatter(z_pca[mask, 0], z_pca[mask, 1], s=15, c=[type_colors[ptype]], 
              alpha=0.5, label=ptype, edgecolor='none', rasterized=True)
ax.set_title(f'PCA (var: {pca.explained_variance_ratio_.sum():.1%})', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
style_legend(ax, loc='best')
add_grid(ax, alpha=0.15)

# Panel 2: t-SNE by type
print("  Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
z_tsne = tsne.fit_transform(all_z)

ax = fig.add_subplot(gs[0, 1])
draw_density_contours(ax, z_tsne[:, 0], z_tsne[:, 1], levels=6, alpha=0.15)
for ptype in ['Source', 'Target', 'Predicted']:
    mask = np.array(point_types) == ptype
    ax.scatter(z_tsne[mask, 0], z_tsne[mask, 1], s=15, c=[type_colors[ptype]],
              alpha=0.5, label=ptype, edgecolor='none', rasterized=True)
ax.set_title('t-SNE', fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
style_legend(ax, loc='best')
add_grid(ax, alpha=0.15)

# Panel 3: UMAP by type
print("  Computing UMAP...")
try:
    from umap import UMAP
    umap_model = UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    z_umap = umap_model.fit_transform(all_z)
    
    ax = fig.add_subplot(gs[0, 2])
    draw_density_contours(ax, z_umap[:, 0], z_umap[:, 1], levels=6, alpha=0.15)
    for ptype in ['Source', 'Target', 'Predicted']:
        mask = np.array(point_types) == ptype
        ax.scatter(z_umap[mask, 0], z_umap[mask, 1], s=15, c=[type_colors[ptype]],
                  alpha=0.5, label=ptype, edgecolor='none', rasterized=True)
    ax.set_title('UMAP', fontsize=12, fontweight='bold')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    style_legend(ax, loc='best')
    add_grid(ax, alpha=0.15)
except ImportError:
    ax = fig.add_subplot(gs[0, 2])
    ax.text(0.5, 0.5, 'UMAP not installed\npip install umap-learn', ha='center', va='center', fontsize=11)
    ax.set_title('UMAP (unavailable)', fontsize=12, fontweight='bold')
    z_umap = None

# Panel 4: PHATE by type
print("  Computing PHATE...")
try:
    import phate
    phate_model = phate.PHATE(n_components=2, random_state=42, n_jobs=1)
    z_phate = phate_model.fit_transform(all_z)
    
    ax = fig.add_subplot(gs[0, 3])
    draw_density_contours(ax, z_phate[:, 0], z_phate[:, 1], levels=6, alpha=0.15)
    for ptype in ['Source', 'Target', 'Predicted']:
        mask = np.array(point_types) == ptype
        ax.scatter(z_phate[mask, 0], z_phate[mask, 1], s=15, c=[type_colors[ptype]],
                  alpha=0.5, label=ptype, edgecolor='none', rasterized=True)
    ax.set_title('PHATE', fontsize=12, fontweight='bold')
    ax.set_xlabel('PHATE 1')
    ax.set_ylabel('PHATE 2')
    style_legend(ax, loc='best')
    add_grid(ax, alpha=0.15)
except ImportError:
    ax = fig.add_subplot(gs[0, 3])
    ax.text(0.5, 0.5, 'PHATE not installed\npip install phate', ha='center', va='center', fontsize=11)
    ax.set_title('PHATE (unavailable)', fontsize=12, fontweight='bold')
    z_phate = None

# ===== ROW 2: Color by STAGE (sources only) with hulls and ellipses =====
z_pca_src = z_pca[:n]
z_tsne_src = z_tsne[:n]

# Panel 5: PCA by stage with hulls
ax = fig.add_subplot(gs[1, 0])
draw_density_contours(ax, z_pca_src[:, 0], z_pca_src[:, 1], levels=6, alpha=0.1)
for stage in stages_list:
    mask = np.array(test_stages) == stage
    color = STAGE_COLORS.get(stage, '#999')
    points = z_pca_src[mask]
    if len(points) > 3:
        draw_convex_hull(ax, points, color, alpha=0.1)
        draw_confidence_ellipse(ax, points[:, 0], points[:, 1], n_std=2.0, color=color, alpha=0.4)
    ax.scatter(points[:, 0], points[:, 1], s=20, c=[color], alpha=0.6, 
              label=stage, edgecolor='white', linewidth=0.3, rasterized=True)
ax.set_title('PCA (by Stage)', fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
style_legend(ax, loc='best', title='Stage')
add_grid(ax, alpha=0.15)

# Panel 6: t-SNE by stage with hulls
ax = fig.add_subplot(gs[1, 1])
draw_density_contours(ax, z_tsne_src[:, 0], z_tsne_src[:, 1], levels=6, alpha=0.1)
for stage in stages_list:
    mask = np.array(test_stages) == stage
    color = STAGE_COLORS.get(stage, '#999')
    points = z_tsne_src[mask]
    if len(points) > 3:
        draw_convex_hull(ax, points, color, alpha=0.1)
    ax.scatter(points[:, 0], points[:, 1], s=20, c=[color], alpha=0.6,
              label=stage, edgecolor='white', linewidth=0.3, rasterized=True)
ax.set_title('t-SNE (by Stage)', fontsize=12, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
style_legend(ax, loc='best', title='Stage')
add_grid(ax, alpha=0.15)

# Panel 7: UMAP by stage with hulls
ax = fig.add_subplot(gs[1, 2])
if z_umap is not None:
    z_umap_src = z_umap[:n]
    draw_density_contours(ax, z_umap_src[:, 0], z_umap_src[:, 1], levels=6, alpha=0.1)
    for stage in stages_list:
        mask = np.array(test_stages) == stage
        color = STAGE_COLORS.get(stage, '#999')
        points = z_umap_src[mask]
        if len(points) > 3:
            draw_convex_hull(ax, points, color, alpha=0.1)
            draw_confidence_ellipse(ax, points[:, 0], points[:, 1], n_std=2.0, color=color, alpha=0.4)
        ax.scatter(points[:, 0], points[:, 1], s=20, c=[color], alpha=0.6,
                  label=stage, edgecolor='white', linewidth=0.3, rasterized=True)
    ax.set_title('UMAP (by Stage)', fontsize=12, fontweight='bold')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    style_legend(ax, loc='best', title='Stage')
    add_grid(ax, alpha=0.15)
else:
    ax.text(0.5, 0.5, 'UMAP not available', ha='center', va='center')

# Panel 8: PHATE by stage with hulls
ax = fig.add_subplot(gs[1, 3])
if z_phate is not None:
    z_phate_src = z_phate[:n]
    draw_density_contours(ax, z_phate_src[:, 0], z_phate_src[:, 1], levels=6, alpha=0.1)
    for stage in stages_list:
        mask = np.array(test_stages) == stage
        color = STAGE_COLORS.get(stage, '#999')
        points = z_phate_src[mask]
        if len(points) > 3:
            draw_convex_hull(ax, points, color, alpha=0.1)
        ax.scatter(points[:, 0], points[:, 1], s=20, c=[color], alpha=0.6,
                  label=stage, edgecolor='white', linewidth=0.3, rasterized=True)
    ax.set_title('PHATE (by Stage)', fontsize=12, fontweight='bold')
    ax.set_xlabel('PHATE 1')
    ax.set_ylabel('PHATE 2')
    style_legend(ax, loc='best', title='Stage')
    add_grid(ax, alpha=0.15)
else:
    ax.text(0.5, 0.5, 'PHATE not available', ha='center', va='center')

# ===== ROW 3: Quality metrics =====
# Panel 9: Variance explained bar (PCA)
ax = fig.add_subplot(gs[2, 0])
n_components_show = min(10, LATENT_DIM)
var_explained = pca.explained_variance_ratio_[:n_components_show] * 100
cumulative = np.cumsum(var_explained)

bars = ax.bar(range(n_components_show), var_explained, color='#6366F1', edgecolor='white', alpha=0.8)
ax.plot(range(n_components_show), cumulative, 'ro-', linewidth=2, markersize=6, label='Cumulative')
ax.axhline(cumulative[-1], color='#EF4444', linestyle='--', alpha=0.5)

ax.set_xlabel('Principal Component', fontsize=10, fontweight='medium')
ax.set_ylabel('Variance Explained (%)', fontsize=10, fontweight='medium')
ax.set_title('PCA Variance Explained', fontsize=11, fontweight='bold')
ax.set_xticks(range(n_components_show))
ax.set_xticklabels([f'PC{i+1}' for i in range(n_components_show)], rotation=45, fontsize=8)
style_legend(ax, loc='upper right')
add_grid(ax, alpha=0.2)

# Panel 10: Silhouette scores by method
ax = fig.add_subplot(gs[2, 1])
from sklearn.metrics import silhouette_score

silhouette_scores = {}
stage_labels_numeric = np.array([stages_list.index(s) if s in stages_list else -1 for s in test_stages])
valid_mask = stage_labels_numeric >= 0

if valid_mask.sum() > 10:
    try:
        silhouette_scores['PCA'] = silhouette_score(z_pca_src[valid_mask], stage_labels_numeric[valid_mask])
        silhouette_scores['t-SNE'] = silhouette_score(z_tsne_src[valid_mask], stage_labels_numeric[valid_mask])
        if z_umap is not None:
            silhouette_scores['UMAP'] = silhouette_score(z_umap_src[valid_mask], stage_labels_numeric[valid_mask])
        if z_phate is not None:
            silhouette_scores['PHATE'] = silhouette_score(z_phate_src[valid_mask], stage_labels_numeric[valid_mask])
    except:
        pass

if silhouette_scores:
    methods = list(silhouette_scores.keys())
    scores = list(silhouette_scores.values())
    colors = ['#6366F1', '#F59E0B', '#10B981', '#EF4444'][:len(methods)]
    
    bars = ax.bar(methods, scores, color=colors, edgecolor='white', width=0.6)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{score:.3f}', ha='center', fontsize=10, fontweight='medium')
    
    ax.axhline(0.5, color='#10B981', linestyle='--', alpha=0.5, label='Good (>0.5)')
    ax.axhline(0.25, color='#F59E0B', linestyle='--', alpha=0.5, label='Fair (>0.25)')
    ax.set_ylabel('Silhouette Score', fontsize=10, fontweight='medium')
    ax.set_title('Stage Separation (Silhouette)', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(scores) * 1.3)
    style_legend(ax, loc='upper right')
else:
    ax.text(0.5, 0.5, 'Silhouette scores unavailable', ha='center', va='center')
add_grid(ax, alpha=0.2)

# Panel 11-12: Inter-method correlation
ax = fig.add_subplot(gs[2, 2:])

# Compute pairwise correlations between embeddings
embed_dict = {'PCA': z_pca_src, 't-SNE': z_tsne_src}
if z_umap is not None:
    embed_dict['UMAP'] = z_umap_src
if z_phate is not None:
    embed_dict['PHATE'] = z_phate_src

n_methods = len(embed_dict)
corr_matrix = np.zeros((n_methods, n_methods))
method_names = list(embed_dict.keys())

for i, m1 in enumerate(method_names):
    for j, m2 in enumerate(method_names):
        # Use distance correlation or simple correlation
        d1 = np.linalg.norm(embed_dict[m1] - embed_dict[m1].mean(axis=0), axis=1)
        d2 = np.linalg.norm(embed_dict[m2] - embed_dict[m2].mean(axis=0), axis=1)
        corr_matrix[i, j] = np.corrcoef(d1, d2)[0, 1]

im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
for i in range(n_methods):
    for j in range(n_methods):
        ax.text(j, i, f'{corr_matrix[i, j]:.2f}', ha='center', va='center',
               fontsize=11, fontweight='medium', color='black' if corr_matrix[i, j] < 0.7 else 'white')

ax.set_xticks(range(n_methods))
ax.set_xticklabels(method_names, fontsize=10)
ax.set_yticks(range(n_methods))
ax.set_yticklabels(method_names, fontsize=10)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Distance Correlation', fontsize=10)
ax.set_title('Embedding Method Agreement', fontsize=11, fontweight='bold')

plt.suptitle('Latent Space Embedding Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig11_embeddings_comparison")
plt.show()

print("\nEmbedding comparison complete!")

In [ ]:
# ============================================================================
# CELL 18: FIGURE - PHATE TRAJECTORY WITH TRANSITIONS (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: PHATE Trajectory Analysis")
print("="*80)

if 'z_phate' in dir() and z_phate is not None:
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)
    
    z_phate_src = z_phate[:n]
    z_phate_tgt = z_phate[n:2*n]
    z_phate_pred = z_phate[2*n:]
    
    # ===== Panel 1: PHATE colored by stage with density and hulls =====
    ax = fig.add_subplot(gs[0, 0])
    
    # Draw overall density
    draw_density_contours(ax, z_phate_src[:, 0], z_phate_src[:, 1], levels=8, alpha=0.15)
    
    # Plot each stage with hull
    for stage in stages_list:
        mask = np.array(test_stages) == stage
        points = z_phate_src[mask]
        color = STAGE_COLORS.get(stage, '#999')
        
        if len(points) > 3:
            draw_convex_hull(ax, points, color, alpha=0.1)
            draw_confidence_ellipse(ax, points[:, 0], points[:, 1], n_std=2.0, 
                                   color=color, alpha=0.4)
        
        ax.scatter(points[:, 0], points[:, 1], s=25, c=[color], alpha=0.6,
                  label=stage, edgecolor='white', linewidth=0.3, rasterized=True)
    
    ax.set_xlabel('PHATE 1', fontsize=11, fontweight='medium')
    ax.set_ylabel('PHATE 2', fontsize=11, fontweight='medium')
    ax.set_title('PHATE: Stage Distribution', fontsize=12, fontweight='bold')
    style_legend(ax, loc='best', title='Stage')
    add_grid(ax, alpha=0.15)
    
    # ===== Panel 2: PHATE with transition arrows (streamlines-like) =====
    ax = fig.add_subplot(gs[0, 1])
    
    # Background: all source points (gray)
    ax.scatter(z_phate_src[:, 0], z_phate_src[:, 1], s=10, alpha=0.15, c='gray')
    
    # Draw transition arrows (sample for clarity)
    n_arrows = min(150, n)
    arrow_idx = np.random.choice(n, n_arrows, replace=False)
    
    # Color arrows by accuracy
    pred_errors = np.linalg.norm(test_preds - test_targets, axis=1)
    
    for i in arrow_idx:
        # Color by prediction error
        err = pred_errors[i]
        err_norm = min(err / (pred_errors.mean() * 2), 1)  # Normalize to [0, 1]
        color = plt.cm.RdYlGn_r(err_norm)
        
        # Draw arrow from source to predicted
        dx = z_phate_pred[i, 0] - z_phate_src[i, 0]
        dy = z_phate_pred[i, 1] - z_phate_src[i, 1]
        ax.arrow(z_phate_src[i, 0], z_phate_src[i, 1], dx * 0.9, dy * 0.9,
                head_width=0.08, head_length=0.04, fc=color, ec=color, alpha=0.6, linewidth=1)
    
    # Add colorbar for error
    sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=plt.Normalize(0, pred_errors.mean() * 2))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label('Prediction Error', fontsize=10)
    
    ax.set_xlabel('PHATE 1', fontsize=11, fontweight='medium')
    ax.set_ylabel('PHATE 2', fontsize=11, fontweight='medium')
    ax.set_title('PHATE: Predicted Transitions', fontsize=12, fontweight='bold')
    add_grid(ax, alpha=0.15)
    
    # ===== Panel 3: True vs Predicted transitions overlay =====
    ax = fig.add_subplot(gs[0, 2])
    
    # Background
    ax.scatter(z_phate_src[:, 0], z_phate_src[:, 1], s=10, alpha=0.1, c='gray')
    
    # Sample arrows
    n_show = min(60, n)
    show_idx = np.random.choice(n, n_show, replace=False)
    
    for i in show_idx:
        # True transition (green, dashed)
        ax.annotate('', xy=z_phate_tgt[i], xytext=z_phate_src[i],
                   arrowprops=dict(arrowstyle='->', color='#10B981', lw=1, alpha=0.4, linestyle='--'))
        # Predicted transition (red)
        ax.annotate('', xy=z_phate_pred[i], xytext=z_phate_src[i],
                   arrowprops=dict(arrowstyle='->', color='#EF4444', lw=1.5, alpha=0.6))
    
    ax.set_xlabel('PHATE 1', fontsize=11, fontweight='medium')
    ax.set_ylabel('PHATE 2', fontsize=11, fontweight='medium')
    ax.set_title('True (green) vs Predicted (red)', fontsize=12, fontweight='bold')
    add_grid(ax, alpha=0.15)
    
    # ===== Panel 4: Prediction error heatmap on PHATE =====
    ax = fig.add_subplot(gs[1, 0])
    
    # Create hexbin
    hb = ax.hexbin(z_phate_src[:, 0], z_phate_src[:, 1], C=pred_errors, 
                   gridsize=25, cmap='YlOrRd', reduce_C_function=np.mean)
    cbar = fig.colorbar(hb, ax=ax, shrink=0.8)
    cbar.set_label('Mean Error', fontsize=10)
    
    ax.set_xlabel('PHATE 1', fontsize=11, fontweight='medium')
    ax.set_ylabel('PHATE 2', fontsize=11, fontweight='medium')
    ax.set_title('Prediction Error (Hexbin)', fontsize=12, fontweight='bold')
    
    # ===== Panel 5: Error by stage violin =====
    ax = fig.add_subplot(gs[1, 1])
    
    violin_data = []
    violin_labels = []
    for stage in stages_list:
        mask = np.array(test_stages) == stage
        if mask.sum() > 0:
            violin_data.append(pred_errors[mask])
            violin_labels.append(stage)
    
    if violin_data:
        parts = ax.violinplot(violin_data, positions=range(len(violin_labels)),
                              showmeans=True, showmedians=True, widths=0.8)
        
        for i, (pc, stage) in enumerate(zip(parts['bodies'], violin_labels)):
            pc.set_facecolor(STAGE_COLORS.get(stage, '#999'))
            pc.set_alpha(0.7)
        
        for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
            if partname in parts:
                parts[partname].set_color('#333')
                parts[partname].set_linewidth(1.5)
    
    ax.set_xticks(range(len(violin_labels)))
    ax.set_xticklabels(violin_labels, fontsize=10)
    ax.set_ylabel('Prediction Error', fontsize=11, fontweight='medium')
    ax.set_xlabel('Stage', fontsize=11, fontweight='medium')
    ax.set_title('Error Distribution by Stage', fontsize=12, fontweight='bold')
    add_grid(ax, alpha=0.2)
    
    # ===== Panel 6: Summary statistics =====
    ax = fig.add_subplot(gs[1, 2])
    ax.axis('off')
    
    # Compute stage-wise statistics
    stage_errors = {}
    for stage in stages_list:
        mask = np.array(test_stages) == stage
        if mask.sum() > 0:
            stage_errors[stage] = pred_errors[mask].mean()
    
    summary_text = f"""
PHATE Trajectory Analysis
{'─' * 35}

Total Transitions: {n:,}

Prediction Error by Stage:
"""
    for stage in stages_list:
        if stage in stage_errors:
            summary_text += f"  {stage}: {stage_errors[stage]:.4f}\n"
    
    best_stage = min(stage_errors, key=stage_errors.get)
    worst_stage = max(stage_errors, key=stage_errors.get)
    
    summary_text += f"""
Best Stage: {best_stage}
  (Error: {stage_errors[best_stage]:.4f})

Worst Stage: {worst_stage}
  (Error: {stage_errors[worst_stage]:.4f})

Overall Error:
  Mean: {pred_errors.mean():.4f}
  Std: {pred_errors.std():.4f}
"""
    
    ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=10,
           verticalalignment='top', fontfamily='monospace',
           bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))
    
    plt.suptitle('PHATE Trajectory Analysis', fontsize=15, fontweight='bold', y=1.02)
    save_figure(fig, OUTPUT_DIR, "fig12_phate_trajectory")
    plt.show()

else:
    print("PHATE not available - install with: pip install phate")

In [ ]:
# ============================================================================
# CELL 19: FIGURE - ATTENTION ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Transformer Attention Analysis")
print("="*80)

# Extract attention weights from model
def get_attention_weights(model, loader, device, max_batches=10):
    """Extract attention weights from transformer layers."""
    model.eval()
    all_attns = []
    
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches:
                break
            
            batch = batch.to(device)
            niche_tokens = batch.niche_tokens
            
            # Try to get attention from niche encoder
            if hasattr(model, 'niche_encoder') and hasattr(model.niche_encoder, 'get_attention_weights'):
                attn = model.niche_encoder.get_attention_weights(niche_tokens)
                if attn is not None:
                    all_attns.append(attn.cpu().numpy())
    
    if all_attns:
        return np.concatenate(all_attns, axis=0)
    return None

# Try to extract attention
try:
    attn_weights = get_attention_weights(model, test_loader, device)
except Exception as e:
    print(f"Could not extract attention: {e}")
    attn_weights = None

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

if attn_weights is not None and len(attn_weights) > 0:
    # Panel 1: Mean attention heatmap
    ax = axes[0, 0]
    mean_attn = attn_weights.mean(axis=0)
    if len(mean_attn.shape) == 2:
        token_names = ['Receiver', 'Ring1', 'Ring2', 'Ring3', 'Ring4', 'HLCA', 'LuCA', 'Pathway', 'Stats'][:mean_attn.shape[0]]
        sns.heatmap(mean_attn, ax=ax, cmap='viridis', 
                    xticklabels=token_names[:mean_attn.shape[1]], 
                    yticklabels=token_names[:mean_attn.shape[0]])
        ax.set_title('Mean Attention Pattern', fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Attention shape unexpected', ha='center', va='center')
else:
    # Fallback: show niche influence analysis
    ax = axes[0, 0]
    
    # Analyze correlation between niche composition and predictions
    niche_influences = cells_df['niche_influence_score'].values[:len(pred_errors)] if 'niche_influence_score' in cells_df.columns else np.zeros(len(pred_errors))
    
    ax.scatter(niche_influences, pred_errors, alpha=0.5, s=15, c='steelblue')
    ax.set_xlabel('Niche Influence Score (Ground Truth)')
    ax.set_ylabel('Prediction Error')
    ax.set_title('Niche Influence vs Prediction Error', fontweight='bold')
    
    # Add correlation
    if len(niche_influences) > 0:
        corr = np.corrcoef(niche_influences, pred_errors)[0, 1]
        ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel 2: Token importance (computed from gradient magnitude)
ax = axes[0, 1]

# Compute importance from input gradients
try:
    model.train()  # Enable gradients
    sample_batch = next(iter(test_loader))
    sample_batch = sample_batch.to(device)
    
    # Create a copy of z_source with gradients enabled
    z_src_grad = sample_batch.z_source.detach().clone().requires_grad_(True)
    
    # We can't easily modify the batch, so let's analyze transition model directly
    # Get context from a forward pass first
    with torch.no_grad():
        outputs = model(sample_batch, return_diagnostics=True)
        context = outputs.get("context", torch.zeros(sample_batch.z_source.size(0), 256, device=device))
    
    # Now compute gradients through transition model
    model.eval()
    t = torch.ones(z_src_grad.size(0), device=device) * 0.5
    edge_ids = torch.zeros(z_src_grad.size(0), dtype=torch.long, device=device)
    
    drift = model.transition_model.forward_drift(
        x_t=z_src_grad,
        t=t,
        context=context.detach(),
        edge_ids=edge_ids
    )
    loss = drift.sum()
    loss.backward()
    
    # Use gradient magnitude as importance proxy
    if z_src_grad.grad is not None:
        importance = z_src_grad.grad.abs().mean(dim=0).cpu().numpy()
        n_dims = min(len(importance), 16)
        ax.bar(range(n_dims), importance[:n_dims], color='coral', edgecolor='black')
        ax.set_xticks(range(n_dims))
        ax.set_xticklabels([f'd{i}' for i in range(n_dims)], rotation=45)
        ax.set_title('Latent Dimension Importance (Gradient)', fontweight='bold')
        ax.set_ylabel('Mean |Gradient|')
    else:
        ax.text(0.5, 0.5, 'Gradients not available', ha='center', va='center')
except Exception as e:
    ax.text(0.5, 0.5, f'Gradient analysis failed:\n{str(e)[:50]}', ha='center', va='center', fontsize=9)
    ax.set_title('Input Importance (unavailable)', fontweight='bold')

# Panel 3: Prediction confidence by stage
ax = axes[1, 0]
stages_unique_test = sorted(set(test_stages))
stage_errors = {stage: pred_errors[np.array(test_stages) == stage] for stage in stages_unique_test}
ax.boxplot([stage_errors[s] for s in stages_unique_test], labels=stages_unique_test)
ax.set_ylabel('Prediction Error (L2)')
ax.set_title('Prediction Error by Stage', fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Panel 4: Error vs transition magnitude
ax = axes[1, 1]
trans_mags = np.linalg.norm(test_targets, axis=1)  # test_targets is already the velocity
ax.scatter(trans_mags, pred_errors, alpha=0.5, s=15, c='purple')
ax.set_xlabel('True Transition Magnitude')
ax.set_ylabel('Prediction Error')
ax.set_title('Error vs Transition Magnitude', fontweight='bold')

# Add trend line
z = np.polyfit(trans_mags, pred_errors, 1)
p = np.poly1d(z)
ax.plot(sorted(trans_mags), p(sorted(trans_mags)), 'r--', linewidth=2, label=f'Trend')
corr = np.corrcoef(trans_mags, pred_errors)[0, 1]
ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig13_attention_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 19: FIGURE - FLOW FIELD RECOVERY (Publication Quality)
# ============================================================================
print("\n" + "="*80)
print("FIGURE: Flow Field Recovery")
print("="*80)

# Compute norms and normalized directions
true_norms = np.linalg.norm(test_targets, axis=1)
pred_norms = np.linalg.norm(test_drifts, axis=1)
true_directions = test_targets / (true_norms[:, None] + 1e-8)
pred_directions = test_drifts / (pred_norms[:, None] + 1e-8)

fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.35, wspace=0.3)

# ===== Panel 1: Direction cosine distribution with violin =====
ax_cos = fig.add_subplot(gs[0, 0])

# Violin plot
parts = ax_cos.violinplot([direction_cosines], positions=[0], showmeans=True, showmedians=True, widths=0.8)
parts['bodies'][0].set_facecolor('#6366F1')
parts['bodies'][0].set_alpha(0.7)
for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
    if partname in parts:
        parts[partname].set_color('#333')
        parts[partname].set_linewidth(1.5)

# Overlay histogram
ax_cos_hist = ax_cos.twinx()
ax_cos_hist.hist(direction_cosines, bins=30, color='#6366F1', edgecolor='white', 
                 alpha=0.3, orientation='horizontal')
ax_cos_hist.set_xlim(ax_cos_hist.get_xlim()[1], 0)  # Flip to left
ax_cos_hist.axis('off')

# Reference lines
ax_cos.axhline(0, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
ax_cos.axhline(0.5, color='#F59E0B', linestyle='--', linewidth=2, label='Good (>0.5)')
ax_cos.axhline(mean_cosine, color='#10B981', linestyle='-', linewidth=2.5, 
              label=f'Mean: {mean_cosine:.3f}')

ax_cos.set_xticks([0])
ax_cos.set_xticklabels(['All Transitions'])
ax_cos.set_ylabel('Cosine Similarity', fontsize=11, fontweight='medium')
ax_cos.set_title('Direction Accuracy Distribution', fontsize=12, fontweight='bold')
ax_cos.set_ylim(-1.1, 1.1)
style_legend(ax_cos, loc='lower right')
add_grid(ax_cos, alpha=0.2)

# ===== Panel 2: Magnitude correlation scatter with regression =====
ax_mag = fig.add_subplot(gs[0, 1])

# Scatter with density coloring
from scipy.stats import gaussian_kde
try:
    xy = np.vstack([true_norms, pred_norms])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    scatter = ax_mag.scatter(true_norms[idx], pred_norms[idx], c=z[idx], s=25, 
                            cmap='viridis', alpha=0.7, edgecolor='none')
    cbar = fig.colorbar(scatter, ax=ax_mag, shrink=0.8)
    cbar.set_label('Density', fontsize=9)
except:
    ax_mag.scatter(true_norms, pred_norms, s=25, alpha=0.5, c='#2563EB', edgecolor='none')

# Perfect prediction line
max_norm = max(true_norms.max(), pred_norms.max()) * 1.1
ax_mag.plot([0, max_norm], [0, max_norm], 'k--', linewidth=2, label='Perfect (y=x)')

# Regression line with CI
z = np.polyfit(true_norms, pred_norms, 1)
p = np.poly1d(z)
x_fit = np.linspace(0, max_norm, 100)
ax_mag.plot(x_fit, p(x_fit), 'r-', linewidth=2.5, 
           label=f'Fit (slope={z[0]:.2f})')

# Correlation annotation
mag_corr = np.corrcoef(true_norms, pred_norms)[0, 1]
ax_mag.text(0.05, 0.95, f'r = {mag_corr:.3f}', transform=ax_mag.transAxes, 
           fontsize=12, fontweight='bold', va='top',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax_mag.set_xlabel('True Transition Magnitude', fontsize=11, fontweight='medium')
ax_mag.set_ylabel('Predicted Transition Magnitude', fontsize=11, fontweight='medium')
ax_mag.set_title('Magnitude Recovery', fontsize=12, fontweight='bold')
ax_mag.set_xlim(0, max_norm)
ax_mag.set_ylim(0, max_norm)
style_legend(ax_mag, loc='lower right')
add_grid(ax_mag, alpha=0.2)

# ===== Panel 3: Direction accuracy by stage (violin) =====
ax_stage = fig.add_subplot(gs[0, 2])

stages_list = [s for s in STAGE_ORDER if s in test_stages]

# Group cosines by stage
stage_cosines = {stage: [] for stage in stages_list}
for i, stage in enumerate(test_stages):
    if stage in stages_list:
        stage_cosines[stage].append(direction_cosines[i])

violin_data = [np.array(stage_cosines[s]) for s in stages_list if len(stage_cosines[s]) > 0]
violin_labels = [s for s in stages_list if len(stage_cosines[s]) > 0]

if violin_data:
    parts = ax_stage.violinplot(violin_data, positions=range(len(violin_labels)),
                                showmeans=True, showmedians=True, widths=0.8)
    
    for i, (pc, stage) in enumerate(zip(parts['bodies'], violin_labels)):
        pc.set_facecolor(STAGE_COLORS.get(stage, '#999'))
        pc.set_alpha(0.7)
    
    for partname in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
        if partname in parts:
            parts[partname].set_color('#333')
            parts[partname].set_linewidth(1.5)
    
    # Add mean annotations
    for i, stage in enumerate(violin_labels):
        mean_val = np.mean(stage_cosines[stage])
        ax_stage.annotate(f'{mean_val:.2f}', (i, mean_val), xytext=(5, 5),
                         textcoords='offset points', fontsize=9, fontweight='medium')
    
    ax_stage.axhline(0.5, color='#F59E0B', linestyle='--', linewidth=1.5, alpha=0.6)
    ax_stage.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.4)

ax_stage.set_xticks(range(len(violin_labels)))
ax_stage.set_xticklabels(violin_labels, fontsize=10)
ax_stage.set_ylabel('Cosine Similarity', fontsize=11, fontweight='medium')
ax_stage.set_xlabel('Stage', fontsize=11, fontweight='medium')
ax_stage.set_title('Direction Accuracy by Stage', fontsize=12, fontweight='bold')
ax_stage.set_ylim(-1.1, 1.1)
add_grid(ax_stage, alpha=0.2)

# ===== Panel 4: Sample flow vectors (2D projection) =====
ax_flow = fig.add_subplot(gs[1, :2])

# Project to 2D
from sklearn.decomposition import PCA
pca_flow = PCA(n_components=2, random_state=42)
src_2d = pca_flow.fit_transform(test_sources)

n_show = min(100, len(test_sources))
sample_idx = np.random.choice(len(test_sources), n_show, replace=False)

# Draw background points (all sources)
ax_flow.scatter(src_2d[:, 0], src_2d[:, 1], s=10, alpha=0.1, c='gray')

# Draw flow arrows
for i in sample_idx:
    # True direction (green)
    true_vec_2d = pca_flow.transform((test_sources[i:i+1] + test_targets[i:i+1]))[0] - src_2d[i]
    pred_vec_2d = pca_flow.transform((test_sources[i:i+1] + test_drifts[i:i+1]))[0] - src_2d[i]
    
    # Scale for visibility
    scale = 0.5
    
    # Color by accuracy
    cos_val = direction_cosines[i]
    color = '#10B981' if cos_val > 0.7 else '#F59E0B' if cos_val > 0.3 else '#EF4444'
    
    # True (dashed, gray)
    ax_flow.arrow(src_2d[i, 0], src_2d[i, 1], 
                 true_vec_2d[0] * scale, true_vec_2d[1] * scale,
                 head_width=0.08, head_length=0.04, fc='gray', ec='gray', 
                 alpha=0.3, linewidth=1, linestyle='--')
    
    # Predicted (solid, colored by accuracy)
    ax_flow.arrow(src_2d[i, 0], src_2d[i, 1],
                 pred_vec_2d[0] * scale, pred_vec_2d[1] * scale,
                 head_width=0.1, head_length=0.05, fc=color, ec=color,
                 alpha=0.7, linewidth=1.5)

# Add legend manually
from matplotlib.patches import Patch, FancyArrow
legend_elements = [
    Patch(facecolor='#10B981', label='Good (cos>0.7)'),
    Patch(facecolor='#F59E0B', label='Fair (0.3-0.7)'),
    Patch(facecolor='#EF4444', label='Poor (<0.3)'),
    Patch(facecolor='gray', label='True direction'),
]
ax_flow.legend(handles=legend_elements, loc='upper right', fontsize=9)

ax_flow.set_xlabel('PC1 (Flow Direction)', fontsize=11, fontweight='medium')
ax_flow.set_ylabel('PC2', fontsize=11, fontweight='medium')
ax_flow.set_title('Predicted vs True Flow (2D Projection)', fontsize=12, fontweight='bold')
add_grid(ax_flow, alpha=0.15)

# ===== Panel 5: Summary metrics =====
ax_summary = fig.add_subplot(gs[1, 2])
ax_summary.axis('off')

# Compute additional metrics
good_pct = (direction_cosines > 0.5).mean() * 100
perfect_pct = (direction_cosines > 0.9).mean() * 100
bad_pct = (direction_cosines < 0).mean() * 100

summary_text = f"""
Flow Field Recovery Summary
{'─' * 35}

Direction Accuracy:
  Mean Cosine: {mean_cosine:.4f}
  Median Cosine: {np.median(direction_cosines):.4f}
  Std Cosine: {np.std(direction_cosines):.4f}

Quality Distribution:
  Excellent (>0.9): {perfect_pct:.1f}%
  Good (>0.5): {good_pct:.1f}%
  Poor (<0): {bad_pct:.1f}%

Magnitude Recovery:
  Correlation: {mag_corr:.4f}
  Mean True: {true_norms.mean():.4f}
  Mean Pred: {pred_norms.mean():.4f}
  Ratio: {pred_norms.mean() / (true_norms.mean() + 1e-8):.3f}

Overall Grade: {'A' if mean_cosine > 0.7 else 'B' if mean_cosine > 0.5 else 'C' if mean_cosine > 0.3 else 'D'}
"""

ax_summary.text(0.05, 0.95, summary_text, transform=ax_summary.transAxes, fontsize=10,
               verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='#F8F9FA', edgecolor='#DEE2E6', linewidth=1.5))

plt.suptitle('Flow Field Recovery Analysis', fontsize=15, fontweight='bold', y=1.02)
save_figure(fig, OUTPUT_DIR, "fig12_flow_recovery")
plt.show()

---
## FINAL SUMMARY

In [ ]:
# ============================================================================
# CELL 20: FINAL SUMMARY AND FIGURE GALLERY
# ============================================================================
print("\n" + "="*80)
print("PIPELINE COMPLETE - FINAL SUMMARY")
print("="*80)

print(f"\nConfiguration:")
print(f"  Difficulty: {DIFFICULTY}")
print(f"  Cells: {N_CELLS}")
print(f"  Donors: {N_DONORS}")
print(f"  Epochs: {N_EPOCHS}")

print(f"\nFinal Metrics:")
print(f"  MSE: {mse:.6f}")
print(f"  MAE: {mae:.6f}")
print(f"  Wasserstein: {w_dist:.6f}")
print(f"  Flow Direction Accuracy: {direction_accuracy:.4f}")

# List all generated figures
print(f"\nGenerated Figures:")
for fig_path in sorted(OUTPUT_DIR.glob("fig*.png")):
    print(f"  {fig_path.name}")

print(f"\nOutput directory: {OUTPUT_DIR}")
print("\n" + "="*80)
print("DONE!")
print("="*80)

In [ ]:
# ============================================================================
# CELL 21: FIGURE GALLERY (display all)
# ============================================================================
print("\n" + "="*80)
print("FIGURE GALLERY")
print("="*80)

from IPython.display import Image

for fig_path in sorted(OUTPUT_DIR.glob("fig*.png")):
    print(f"\n{fig_path.stem}")
    print("-" * 60)
    display(Image(filename=str(fig_path), width=800))